## 6-SKU Case Code and Output

In [1]:
#!/usr/bin/env python3
"""
================================================================================
RAMP-ML — 6 SKU "AM vince per lead time" — CON COSTO BACKORDER ANALITICO
================================================================================
"""

import numpy as np
import gurobipy as gp
from gurobipy import GRB
from scipy.stats import poisson
from itertools import combinations
import time


def expected_backorder(mu_L, beta):
    """E[(D_L - beta)+] for D_L ~ Poisson(mu_L). Identica alla versione CG."""
    if mu_L < 1e-9 or beta < 0:
        return mu_L
    if beta == 0:
        return mu_L
    return max(mu_L*(1-poisson.cdf(beta-1, mu_L)) - beta*(1-poisson.cdf(beta, mu_L)), 0.0)


def solve_stochastic(
    nI, nD, nC, nS,
    mu_ic, alpha_ic,
    SL, SL_raw,
    c_CM, o, h_d, b_i, L_CM,
    n_i, c_raw, den_raw, unit_raw,
    o_raw, L_raw, L_AM, Leas, Cap_3DP, a_i,
    t_central,
    t_dic=None, B_cap=None, costo_km=100.0,
    coord_DC=None, coord_C=None,
    L_card=None, U_card=None,
    mip_gap=0.01, time_limit=600,
    xi=2.0, seed=42,
    warm_start_sol=None,
    verbose=False, scenario_name="default",
    include_backorder_LT=True,   # <-- TOGGLE per confronto A/B
):
    t_start = time.time()
    I, D, C, S = range(nI), range(nD), range(nC), range(nS)
    pi_s = 1.0 / nS
    SL_i = np.full(nI, SL) if np.isscalar(SL) else np.asarray(SL, dtype=float)

    if t_dic is None and coord_DC is not None and coord_C is not None:
        dist = np.array([
            [np.sqrt((coord_DC[d,0]-coord_C[c,0])**2 +
                     (coord_DC[d,1]-coord_C[c,1])**2)
             for c in range(nC)] for d in range(nD)])
        t_dic = np.stack([dist * costo_km] * nI, axis=1)

    # ── Preprocessing ─────────────────────────────────────────────────────
    prod_i  = n_i * c_CM
    vol_i   = (prod_i / 1.30) * 1e-6
    qbar_i  = (vol_i * den_raw) / unit_raw
    ptime_i = prod_i / (1.30 * 0.00525 * 3600)

    def beta_poisson(mu_val, L_val, SL_target):
        lam = mu_val * L_val
        if lam <= 1e-9: return 0
        for s in range(100000):
            if poisson.cdf(s, lam) >= SL_target: return int(s)
        return 99999

    mu_grid_set = {0.0}
    for i in I:
        for r in range(1, nC + 1):
            for combo in combinations([float(mu_ic[i,c]) for c in C], r):
                mu_grid_set.add(round(sum(combo), 6))
    mu_grid = np.array(sorted(mu_grid_set))
    M = len(mu_grid)

    B_CM_grid  = np.zeros((nI, nD, M))
    B_AM_grid  = np.zeros((nI, nD, M))
    C_EOQ_grid = np.zeros((nI, nD, M))
    for i in I:
        for d in D:
            for m, mu_m in enumerate(mu_grid):
                B_CM_grid[i,d,m] = beta_poisson(mu_m, L_CM[i], SL_i[i])
                B_AM_grid[i,d,m] = beta_poisson(mu_m, L_AM[i,d], SL_i[i])
                if mu_m > 0:
                    C_EOQ_grid[i,d,m] = np.sqrt(2*o*h_d[d]*c_CM[i]*mu_m)

    B_bar = B_CM_grid[:,:,-1]
    mu_id_tot = mu_ic.sum(axis=1)
    mu_max_i = np.array([mu_ic[i,:].sum() for i in I])

    contributi_raw = [float(qbar_i[i]*mu_id_tot[i]) for i in I]
    if nI <= 15:
        mu_raw_grid_set = {0.0}
        for r in range(1, nI+1):
            for combo in combinations(contributi_raw, r):
                mu_raw_grid_set.add(round(sum(combo), 4))
        mu_raw_grid = np.array(sorted(mu_raw_grid_set))
    else:
        mu_raw_grid = np.linspace(0, sum(contributi_raw), 200)
    M_RAW = len(mu_raw_grid)
    B_raw_grid = np.zeros((nD, M_RAW))
    C_EOQ_raw_grid = np.zeros((nD, M_RAW))
    B_bar_raw = np.zeros(nD)
    for d in D:
        for m, mu_m in enumerate(mu_raw_grid):
            B_raw_grid[d,m] = beta_poisson(mu_m, L_raw, SL_raw)
            if mu_m > 0:
                C_EOQ_raw_grid[d,m] = np.sqrt(2*o_raw*h_d[d]*c_raw*mu_m)
        B_bar_raw[d] = B_raw_grid[d,-1]

    rng = np.random.default_rng(seed)
    d_ics = rng.poisson(mu_ic[:,:,np.newaxis], size=(nI,nC,nS)).astype(float)

    Q_bar_CM = np.zeros((nI,nD)); F_bar = np.zeros((nI,nD))
    for i in I:
        for d in D:
            Q_bar_CM[i,d] = d_ics[i,:,:].sum(axis=0).max() * xi
            F_bar[i,d] = d_ics[i,:,:].max() * xi
    Q_bar_AM = np.full((nI,nD), Cap_3DP * xi)

    if L_card is None: L_card = np.ones(nI, dtype=int)
    if U_card is None: U_card = np.full(nI, nD, dtype=int)

    # ── PWL breakpoints (comuni a beta_aux e ai nuovi bo_aux) ─────────────
    N_PWL = 30
    pwl_mu_pts = {}
    for i in I:
        pwl_mu_pts[i] = np.linspace(0.0, max(mu_max_i[i], 0.01), N_PWL)

    # ── Modello ───────────────────────────────────────────────────────────
    mdl = gp.Model(f"stoc_{scenario_name}")
    mdl.setParam("OutputFlag", 1 if verbose else 0)
    mdl.setParam("MIPGap", mip_gap)
    mdl.setParam("TimeLimit", time_limit)

    am   = mdl.addVars(nI, vtype=GRB.BINARY, name="am")
    cm   = mdl.addVars(nI, vtype=GRB.BINARY, name="cm")
    y    = mdl.addVars(nD, nI, nC, vtype=GRB.BINARY, name="y")
    dc   = mdl.addVars(nD, nI, vtype=GRB.BINARY, name="dc")
    v    = mdl.addVars(nI, nD, vtype=GRB.BINARY, name="v")
    w    = mdl.addVars(nI, nD, vtype=GRB.BINARY, name="w")
    B    = mdl.addVars(nI, nD, lb=0.0, name="B")
    n3DP = mdl.addVars(nD, vtype=GRB.INTEGER, lb=0, name="n3DP")
    mu_var = mdl.addVars(nD, nI, lb=0.0, name="mu_var")
    C_EOQ_var = mdl.addVars(nD, nI, lb=0.0, name="C_EOQ_var")
    dcAM = mdl.addVars(nD, vtype=GRB.BINARY, name="dcAM")
    C_EOQ_raw_var = mdl.addVars(nD, lb=0.0, name="C_EOQ_raw_var")
    B_raw = mdl.addVars(nD, lb=0.0, name="B_raw")
    mu_raw_contrib = mdl.addVars(nD, nI, lb=0.0, name="mu_raw_contrib")
    mu_raw_total = mdl.addVars(nD, lb=0.0, name="mu_raw_total")
    beta_cm_aux = mdl.addVars(nI, nD, lb=0.0, name="beta_cm_aux")
    beta_am_aux = mdl.addVars(nI, nD, lb=0.0, name="beta_am_aux")

    # ══════════════════════════════════════════════════════════════════════
    # [NUOVO] Variabili ausiliarie PWL per il costo di backorder analitico
    # ══════════════════════════════════════════════════════════════════════
    bo_cm_aux = mdl.addVars(nI, nD, lb=0.0, name="bo_cm_aux")
    bo_am_aux = mdl.addVars(nI, nD, lb=0.0, name="bo_am_aux")

    q_CM  = mdl.addVars(nI, nD, nS, lb=0.0, name="q_CM")
    q_AM  = mdl.addVars(nI, nD, nS, lb=0.0, name="q_AM")
    f     = mdl.addVars(nI, nD, nC, nS, lb=0.0, name="f")
    u     = mdl.addVars(nI, nC, nS, lb=0.0, name="u")
    I_inv = mdl.addVars(nI, nD, nS, lb=0.0, name="I_inv")

    # ── Vincoli primo stadio ──────────────────────────────────────────────
    mdl.addConstrs((am[i]+cm[i]==1 for i in I), name="A1")
    mdl.addConstrs((y[d,i,c]<=dc[d,i] for d in D for i in I for c in C), name="B1")
    mdl.addConstrs((dc[d,i]<=gp.quicksum(y[d,i,c] for c in C) for d in D for i in I), name="B2")
    mdl.addConstrs((gp.quicksum(y[d,i,c] for d in D)==1 for i in I for c in C), name="assign")
    mdl.addConstrs((gp.quicksum(dc[d,i] for d in D)>=L_card[i] for i in I), name="B3_lo")
    mdl.addConstrs((gp.quicksum(dc[d,i] for d in D)<=U_card[i] for i in I), name="B3_up")
    mdl.addConstrs((v[i,d]<=am[i] for i in I for d in D), name="D1")
    mdl.addConstrs((v[i,d]<=dc[d,i] for i in I for d in D), name="D2")
    mdl.addConstrs((v[i,d]>=am[i]+dc[d,i]-1 for i in I for d in D), name="D2b")
    mdl.addConstrs((w[i,d]<=cm[i] for i in I for d in D), name="W1")
    mdl.addConstrs((w[i,d]<=dc[d,i] for i in I for d in D), name="W2")
    mdl.addConstrs((w[i,d]>=cm[i]+dc[d,i]-1 for i in I for d in D), name="W3")

    mdl.addConstrs((mu_var[d,i]==gp.quicksum(mu_ic[i,c]*y[d,i,c] for c in C)
                    for d in D for i in I), name="DG2_direct")
    mdl.addConstrs((mu_var[d,i]<=mu_max_i[i]*dc[d,i] for d in D for i in I), name="DG2_dc")

    for i in I:
        pts = pwl_mu_pts[i].tolist()
        for d in D:
            bcm = [float(beta_poisson(mu, L_CM[i], SL_i[i])) for mu in pts]
            bam = [float(beta_poisson(mu, L_AM[i,d], SL_i[i])) for mu in pts]
            eoq = [float(np.sqrt(2*o*h_d[d]*c_CM[i]*mu)) if mu>0 else 0.0 for mu in pts]
            mdl.addGenConstrPWL(mu_var[d,i], beta_cm_aux[i,d], pts, bcm, name=f"PWL_bCM_{d}_{i}")
            mdl.addGenConstrPWL(mu_var[d,i], beta_am_aux[i,d], pts, bam, name=f"PWL_bAM_{d}_{i}")
            mdl.addGenConstrPWL(mu_var[d,i], C_EOQ_var[d,i], pts, eoq, name=f"PWL_EOQ_{d}_{i}")

            bo_cm_pts = [b_i[i] * expected_backorder(pts[m]*L_CM[i], bcm[m])
                         for m in range(len(pts))]
            bo_am_pts = [b_i[i] * expected_backorder(pts[m]*L_AM[i,d], bam[m])
                         for m in range(len(pts))]
            mdl.addGenConstrPWL(mu_var[d,i], bo_cm_aux[i,d], pts, bo_cm_pts,
                                name=f"PWL_boCM_{d}_{i}")
            mdl.addGenConstrPWL(mu_var[d,i], bo_am_aux[i,d], pts, bo_am_pts,
                                name=f"PWL_boAM_{d}_{i}")

    mdl.addConstrs((B[i,d]>=beta_cm_aux[i,d]-B_bar[i,d]*(1-cm[i]) for i in I for d in D), name="DG4a")
    mdl.addConstrs((B[i,d]>=beta_am_aux[i,d]-B_bar[i,d]*(1-am[i]) for i in I for d in D), name="DG4b")
    mdl.addConstrs((B[i,d]<=B_bar[i,d]*dc[d,i] for i in I for d in D), name="DG5")

    if B_cap is not None:
        mdl.addConstrs((gp.quicksum(B[i,d] for i in I)<=B_cap for d in D), name="DG5b")

    for d in D:
        for i in I: mdl.addConstr(dcAM[d]>=v[i,d], name=f"dcAM_lo_{d}_{i}")
        mdl.addConstr(dcAM[d]<=gp.quicksum(v[i,d] for i in I), name=f"dcAM_up_{d}")

    M_raw_big = float(max(qbar_i)*sum(mu_ic[i,:].sum() for i in I))*2
    for d in D:
        for i in I:
            mdl.addConstr(mu_raw_contrib[d,i]<=M_raw_big*v[i,d], name=f"RC1_{d}_{i}")
            mdl.addConstr(mu_raw_contrib[d,i]<=qbar_i[i]*mu_var[d,i], name=f"RC2_{d}_{i}")
            mdl.addConstr(mu_raw_contrib[d,i]>=qbar_i[i]*mu_var[d,i]-M_raw_big*(1-v[i,d]), name=f"RC3_{d}_{i}")
        mdl.addConstr(mu_raw_total[d]==gp.quicksum(mu_raw_contrib[d,i] for i in I), name=f"mu_raw_tot_{d}")
        mdl.addConstr(mu_raw_total[d]<=M_raw_big*dcAM[d], name=f"mu_raw_dc_{d}")

    mu_raw_max = float(sum(qbar_i[i]*mu_ic[i,:].sum() for i in I))
    mu_raw_pts = np.linspace(0.0, max(mu_raw_max,0.01), 30).tolist()
    for d in D:
        brp = [float(beta_poisson(mu, L_raw, SL_raw)) for mu in mu_raw_pts]
        erp = [float(np.sqrt(2*o_raw*h_d[d]*c_raw*mu)) if mu>0 else 0.0 for mu in mu_raw_pts]
        bra = mdl.addVar(lb=0.0, name=f"b_raw_aux_{d}")
        mdl.addGenConstrPWL(mu_raw_total[d], bra, mu_raw_pts, brp, name=f"PWL_Braw_{d}")
        mdl.addConstr(B_raw[d]>=bra, name=f"ER4_{d}")
        mdl.addConstr(B_raw[d]<=B_bar_raw[d]*dcAM[d], name=f"ER5_{d}")
        mdl.addGenConstrPWL(mu_raw_total[d], C_EOQ_raw_var[d], mu_raw_pts, erp, name=f"PWL_EOQraw_{d}")

    M_EOQ_big = float(C_EOQ_grid.max())*2
    C_EOQ_act = mdl.addVars(nD, nI, lb=0.0, name="C_EOQ_act")
    for d in D:
        for i in I:
            mdl.addConstr(C_EOQ_act[d,i]<=M_EOQ_big*w[i,d], name=f"L_EOQ1_{d}_{i}")
            mdl.addConstr(C_EOQ_act[d,i]<=C_EOQ_var[d,i], name=f"L_EOQ2_{d}_{i}")
            mdl.addConstr(C_EOQ_act[d,i]>=C_EOQ_var[d,i]-M_EOQ_big*(1-w[i,d]), name=f"L_EOQ3_{d}_{i}")

    M_BO_big = max(
        max(b_i[i] * mu_max_i[i] for i in I),
        1.0
    ) * 2
    BO_act = mdl.addVars(nD, nI, lb=0.0, name="BO_act")
    for d in D:
        for i in I:
            mdl.addConstr(BO_act[d,i] <= M_BO_big*cm[i] + M_BO_big*am[i], name=f"BO_cap_{d}_{i}")
            mdl.addConstr(BO_act[d,i] >= bo_cm_aux[i,d] - M_BO_big*(1-cm[i]), name=f"BO_lo_CM_{d}_{i}")
            mdl.addConstr(BO_act[d,i] >= bo_am_aux[i,d] - M_BO_big*(1-am[i]), name=f"BO_lo_AM_{d}_{i}")
            mdl.addConstr(BO_act[d,i] <= bo_cm_aux[i,d] + M_BO_big*(1-cm[i]), name=f"BO_up_CM_{d}_{i}")
            mdl.addConstr(BO_act[d,i] <= bo_am_aux[i,d] + M_BO_big*(1-am[i]), name=f"BO_up_AM_{d}_{i}")
            mdl.addConstr(BO_act[d,i] <= M_BO_big*dc[d,i], name=f"BO_dc_{d}_{i}")

    ratio_pt = np.where(qbar_i>1e-9, ptime_i/qbar_i, 0.0)
    mdl.addConstrs((n3DP[d]>=gp.quicksum(ratio_pt[i]*mu_raw_contrib[d,i] for i in I)/Cap_3DP for d in D), name="E1")

    # ══════════════════════════════════════════════════════════════════════
    # McCormick per tecnologia-specifico Base Stock
    # ══════════════════════════════════════════════════════════════════════
    B_cm_lin = mdl.addVars(nI, nD, lb=0.0, name="B_cm_lin")
    B_am_lin = mdl.addVars(nI, nD, lb=0.0, name="B_am_lin")
    for i in I:
        for d in D:
            Bub = float(B_bar[i,d])
            mdl.addConstr(B_cm_lin[i,d] <= Bub * w[i,d])
            mdl.addConstr(B_cm_lin[i,d] <= B[i,d])
            mdl.addConstr(B_cm_lin[i,d] >= B[i,d] - Bub*(1-w[i,d]))
            mdl.addConstr(B_am_lin[i,d] <= Bub * v[i,d])
            mdl.addConstr(B_am_lin[i,d] <= B[i,d])
            mdl.addConstr(B_am_lin[i,d] >= B[i,d] - Bub*(1-v[i,d]))

    cost_BS = gp.quicksum(c_CM[i]*(1+h_d[d])*B_cm_lin[i,d] + prod_i[i]*(1+h_d[d])*B_am_lin[i,d] for i in I for d in D)
    cost_EOQ = gp.quicksum(C_EOQ_act[d,i] for i in I for d in D)
    # cost_raw RIMOSSO: consumo diretto materia prima gia' incluso in prod_i,
    # contarlo separatamente era doppio conteggio. Restano EOQ_raw/holding_raw.
    cost_raw_dc = gp.quicksum(C_EOQ_raw_var[d]+h_d[d]*c_raw*B_raw[d] for d in D)
    cost_leas = gp.quicksum(Leas*n3DP[d] for d in D)

    if include_backorder_LT:
        cost_backorder_LT = gp.quicksum(BO_act[d,i] for i in I for d in D)
    else:
        cost_backorder_LT = 0.0

    # Secondo stadio (invariato)
    mdl.addConstrs((gp.quicksum(a_i[i]*q_AM[i,d,s] for i in I)<=Cap_3DP*n3DP[d] for d in D for s in S), name="E2")
    mdl.addConstrs((B[i,d]+q_CM[i,d,s]+q_AM[i,d,s]-gp.quicksum(f[i,d,c,s] for c in C)==I_inv[i,d,s]
                    for i in I for d in D for s in S), name="F1")
    mdl.addConstrs((gp.quicksum(f[i,d,c,s] for d in D)+u[i,c,s]==d_ics[i,c,s]
                    for i in I for c in C for s in S), name="F2")
    mdl.addConstrs((f[i,d,c,s]<=F_bar[i,d]*dc[d,i] for i in I for d in D for c in C for s in S), name="G1")
    mdl.addConstrs((q_CM[i,d,s]<=Q_bar_CM[i,d]*w[i,d] for i in I for d in D for s in S), name="G2")
    mdl.addConstrs((q_AM[i,d,s]<=Q_bar_AM[i,d]*v[i,d] for i in I for d in D for s in S), name="G3")
    mdl.addConstrs((gp.quicksum(pi_s*gp.quicksum(f[i,d,c,s] for d in D) for s in S)
                    >=alpha_ic[i,c]*gp.quicksum(pi_s*d_ics[i,c,s] for s in S)
                    for i in I for c in C), name="H")

    cost_2nd = gp.quicksum(pi_s*(
        gp.quicksum(c_CM[i]*q_CM[i,d,s] for i in I for d in D)+
        gp.quicksum(prod_i[i]*q_AM[i,d,s] for i in I for d in D)+
        gp.quicksum(t_dic[d,i,c]*f[i,d,c,s] for i in I for d in D for c in C)+
        gp.quicksum(h_d[d]*I_inv[i,d,s] for i in I for d in D)+
        gp.quicksum(b_i[i]*u[i,c,s] for i in I for c in C)
    ) for s in S)

    mdl.setObjective(cost_BS+cost_EOQ+cost_raw_dc+cost_leas
                     +cost_backorder_LT+cost_2nd, GRB.MINIMIZE)

    # VarHint
    from scipy.stats import poisson as _pois
    def _bp(mu, L, sl):
        lam = mu*L
        if lam<1e-9: return 0
        for ss in range(10000):
            if _pois.cdf(ss, lam)>=sl: return ss
        return 9999
    for i in I:
        mu_c = float(mu_ic[i,:].sum())
        b_cm_h, b_am_h = _bp(mu_c, L_CM[i], SL_i[i]), _bp(mu_c, L_AM[i,0], SL_i[i])
        risp = c_CM[i]*(1+h_d[0])*(b_cm_h-b_am_h)
        bo_cm = b_i[i]*expected_backorder(mu_c*L_CM[i], b_cm_h)
        bo_am = b_i[i]*expected_backorder(mu_c*L_AM[i,0], b_am_h)
        risp += (bo_cm - bo_am) if include_backorder_LT else 0.0
        costo = (prod_i[i]-c_CM[i])*mu_c
        netto = risp - costo
        if netto>0:
            am[i].VarHintVal=1.0; am[i].VarHintPri=10; cm[i].VarHintVal=0.0
        elif -Leas<netto<=0:
            am[i].VarHintVal=1.0; am[i].VarHintPri=5; cm[i].VarHintVal=0.0
        else:
            am[i].VarHintVal=0.0; am[i].VarHintPri=1; cm[i].VarHintVal=1.0
    mdl.update()

    mdl.optimize()
    solve_time = time.time() - t_start

    if mdl.Status not in [GRB.OPTIMAL, GRB.TIME_LIMIT, GRB.SUBOPTIMAL]:
        return {'status':'INFEASIBLE','obj':None,'obj_real':None,'gap':None,
                'time':solve_time,'am':None,'cm':None,'dc':None,'B':None,
                'n3DP':None,'cost_breakdown':None,'y_sol':None}

    status = {GRB.OPTIMAL:'OPTIMAL',GRB.TIME_LIMIT:'TIME_LIMIT',
              GRB.SUBOPTIMAL:'SUBOPTIMAL'}.get(mdl.Status, str(mdl.Status))

    def snap(val): return val if val>0.5 else 0.0
    am_sol = np.array([am[i].X>0.5 for i in I])
    cm_sol = np.array([cm[i].X>0.5 for i in I])
    dc_sol = np.array([[dc[d,i].X>0.5 for i in I] for d in D])
    B_sol  = np.array([[B[i,d].X for d in D] for i in I])
    n3DP_sol = np.array([max(0.0, n3DP[d].X) for d in D])
    w_sol = np.array([[w[i,d].X for d in D] for i in I])
    v_sol = np.array([[v[i,d].X for d in D] for i in I])
    y_sol = np.array([[[snap(y[d,i,c].X) for c in C] for i in I] for d in D])
    n_dc_per_sku = dc_sol.sum(axis=0)

    bo_lt_sol = sum(BO_act[d,i].X for i in I for d in D)

    mu_var_sol = np.array([[sum(mu_ic[i,c]*y_sol[d,i,c] for c in C) for i in I] for d in D])
    C_EOQ_act_sol = np.zeros((nD,nI))
    for d in D:
        for i in I:
            if w_sol[i,d]>0.5:
                m = int(np.argmin(np.abs(mu_grid-mu_var_sol[d,i])))
                C_EOQ_act_sol[d,i] = C_EOQ_grid[i,d,m]
    mu_raw_sol = np.array([[qbar_i[i]*mu_var_sol[d,i]*float(v_sol[i,d]>0.5) for i in I] for d in D])
    dcAM_sol = np.array([any(v_sol[i,d]>0.5 for i in I) for d in D])
    C_EOQ_raw_sol = np.zeros(nD); B_raw_sol_arr = np.zeros(nD)
    for d in D:
        if dcAM_sol[d]:
            mu_raw_d = mu_raw_sol[d,:].sum()
            m = int(np.argmin(np.abs(mu_raw_grid-mu_raw_d)))
            C_EOQ_raw_sol[d] = C_EOQ_raw_grid[d,m]; B_raw_sol_arr[d] = B_raw_grid[d,m]

    # FIX: base-stock tecnologia-specifico anche nel reporting (prima usava
    # sempre c_CM anche per gli SKU AM, disallineato dal vero obiettivo
    # che usa B_cm_lin/B_am_lin). w_sol/v_sol selezionano il ramo corretto.
    cost_BS_sol = sum(
        c_CM[i]*(1+h_d[d])*B_sol[i,d] if w_sol[i,d] > 0.5
        else (prod_i[i]*(1+h_d[d])*B_sol[i,d] if v_sol[i,d] > 0.5 else 0.0)
        for i in I for d in D
    )
    cost_EOQ_sol = sum(C_EOQ_act_sol[d,i] for d in D for i in I)
    # cost_raw_sol RIMOSSO (stesso motivo del fix nell'obiettivo)
    cost_raw_dc_sol = sum(C_EOQ_raw_sol[d]+h_d[d]*c_raw*B_raw_sol_arr[d] for d in D)
    cost_leas_sol = sum(Leas*n3DP_sol[d] for d in D)
    cost_1st = cost_BS_sol+cost_EOQ_sol+cost_raw_dc_sol+cost_leas_sol+bo_lt_sol

    pp = gp.Model("pp"); pp.setParam("OutputFlag",0)
    qp = pp.addVars(nI,nD,nS,lb=0.0); qa = pp.addVars(nI,nD,nS,lb=0.0)
    fp = pp.addVars(nI,nD,nC,nS,lb=0.0); up = pp.addVars(nI,nC,nS,lb=0.0)
    ip = pp.addVars(nI,nD,nS,lb=0.0)
    pp.addConstrs((gp.quicksum(a_i[i]*qa[i,d,s] for i in I)<=Cap_3DP*n3DP_sol[d] for d in D for s in S))
    pp.addConstrs((B_sol[i,d]+qp[i,d,s]+qa[i,d,s]-gp.quicksum(fp[i,d,c,s] for c in C)==ip[i,d,s]
                   for i in I for d in D for s in S))
    pp.addConstrs((gp.quicksum(fp[i,d,c,s] for d in D)+up[i,c,s]==d_ics[i,c,s]
                   for i in I for c in C for s in S))
    pp.addConstrs((fp[i,d,c,s]<=F_bar[i,d]*float(dc_sol[d,i]) for i in I for d in D for c in C for s in S))
    pp.addConstrs((qp[i,d,s]<=Q_bar_CM[i,d]*float(w_sol[i,d]>0.5) for i in I for d in D for s in S))
    pp.addConstrs((qa[i,d,s]<=Q_bar_AM[i,d]*float(v_sol[i,d]>0.5) for i in I for d in D for s in S))
    pp.setObjective(gp.quicksum(pi_s*(
        gp.quicksum(c_CM[i]*qp[i,d,s] for i in I for d in D)+
        gp.quicksum(prod_i[i]*qa[i,d,s] for i in I for d in D)+
        gp.quicksum(t_dic[d,i,c]*fp[i,d,c,s] for i in I for d in D for c in C)+
        gp.quicksum(h_d[d]*ip[i,d,s] for i in I for d in D)+
        gp.quicksum(b_i[i]*up[i,c,s] for i in I for c in C)) for s in S), GRB.MINIMIZE)
    pp.optimize()

    if pp.Status==GRB.OPTIMAL:
        cost_breakdown = {
            'base_stock': cost_BS_sol, 'EOQ_CM': cost_EOQ_sol,
            'EOQ_raw': sum(C_EOQ_raw_sol), 'holding_raw': sum(h_d[d]*c_raw*B_raw_sol_arr[d] for d in D),
            'leasing_3DP': cost_leas_sol,
            'backorder_LT_analitico': bo_lt_sol,
            'trasporto': sum(pi_s*t_dic[d,i,c]*fp[i,d,c,s].X for i in I for d in D for c in C for s in S),
            'produzione_CM': sum(pi_s*c_CM[i]*qp[i,d,s].X for i in I for d in D for s in S),
            'produzione_AM': sum(pi_s*prod_i[i]*qa[i,d,s].X for i in I for d in D for s in S),
            'holding_inv': sum(pi_s*h_d[d]*ip[i,d,s].X for i in I for d in D for s in S),
            'backorder_2ndstage': sum(pi_s*b_i[i]*up[i,c,s].X for i in I for c in C for s in S),
        }
        obj_real = cost_1st+pp.ObjVal
        # Controllo di coerenza: il totale ricalcolato dalla scomposizione
        # deve combaciare con l'obiettivo vero del solver (entro tolleranza
        # numerica). Se questo stampa un warning, c'e' un altro disallineamento
        # nella scomposizione da investigare prima di fidarsi dei numeri.
        if abs(obj_real - mdl.ObjVal) > 1.0:
            print(f"  [ATTENZIONE] obj_real ({obj_real:,.2f}) != mdl.ObjVal "
                  f"({mdl.ObjVal:,.2f}), diff={obj_real-mdl.ObjVal:,.2f}€")
    else:
        cost_breakdown = {}
        obj_real = mdl.ObjVal

    return {'status':status,'obj':mdl.ObjVal,'obj_real':obj_real,
            'gap':mdl.MIPGap*100,'time':solve_time,
            'am':am_sol,'cm':cm_sol,'dc':dc_sol,'B':B_sol,
            'n3DP':n3DP_sol,'n_dc_per_sku':n_dc_per_sku,
            'cost_breakdown':cost_breakdown,'y_sol':y_sol}


def print_results(res, name, nI, nD, c_CM, c_AM, b_i, L_CM, L_AM):
    print(f"\n{'='*70}\nRISULTATI: {name}\n{'='*70}")
    if res['status']=='INFEASIBLE': print("  INFEASIBLE"); return
    print(f"  Status: {res['status']}  |  Costo: {res['obj_real']:,.2f}€  |  "
          f"Gap: {res['gap']:.2f}%  |  Tempo: {res['time']:.1f}s")
    print(f"\n  {'SKU':<4} {'T':<3} {'DC':<10} {'B':>4} {'c_CM':>7} {'c_AM':>7} {'ratio':>6}")
    print(f"  {'-'*55}")
    for i in range(nI):
        t = "AM" if res['am'][i] else "CM"
        dcs = [d for d in range(nD) if res['dc'][d,i]]
        bm = res['B'][i,dcs].mean() if dcs else 0
        print(f"  {i:<4} {t:<3} {str(dcs):<10} {bm:>4.1f} {c_CM[i]:>7.0f} "
              f"{c_AM[i]:>7.0f} {c_AM[i]/c_CM[i]:>6.2f}")
    print(f"\n  Stampanti: {np.round(res['n3DP'],1)}")
    if res.get('cost_breakdown'):
        print(f"  Costi:")
        tot = 0.0
        for k,v in res['cost_breakdown'].items():
            if abs(v)>0.01:
                print(f"    {k:<24}: {v:>12.2f}€")
                tot += v
        print(f"    {'TOTALE':<24}: {tot:>12.2f}€")


# ══════════════════════════════════════════════════════════════════════════
# CASO 6 SKU — "AM vince per lead time, non per prezzo"
# ══════════════════════════════════════════════════════════════════════════
nI, nD, nC = 6, 3, 5
seed = 42

coord_DC = np.array([[1.0,8.0],[1.0,2.0],[4.0,5.0]])
coord_C  = np.array([[1.5,9.0],[0.5,7.5],[1.5,1.0],[0.5,2.5],[4.5,5.5]])

c_CM = np.array([5000.,5000., 1000.,1000., 200.,200.])
c_AM = np.array([6000.,6000., 1500.,1500., 800.,800.])
n_i  = c_AM / c_CM

L_CM = np.array([26,26, 26,26, 9,9]) / 52
L_AM_vec = np.array([1,1, 1,1, 2,2]) / 52
L_AM = np.column_stack([L_AM_vec] * nD)

b_i  = np.array([150000.,120000., 50000.,40000., 3000.,2000.])
SL_i = np.array([0.99,0.99, 0.99,0.99, 0.90,0.88])

mu_ic = np.array([
    [0.8,0.7,0.5,0.5,0.5],
    [0.6,0.6,0.6,0.6,0.6],
    [0.5,0.4,0.4,0.4,0.3],
    [0.4,0.4,0.4,0.4,0.4],
    [2.0,2.0,2.0,2.0,2.0],
    [2.0,2.0,2.0,2.0,2.0],
])
alpha_ic = np.column_stack([SL_i] * nC)
a_i = c_AM / (1.30 * 0.00525 * 3600)
h_d = np.full(nD, 0.20)

costo_km = 50.0
Leas     = 15000.0

dist = np.array([[np.sqrt((coord_DC[d,0]-coord_C[c,0])**2 +
                          (coord_DC[d,1]-coord_C[c,1])**2)
                  for c in range(nC)] for d in range(nD)])
t_dic = np.stack([dist * costo_km] * nI, axis=1)

kwargs = dict(nI=nI, nD=nD, nC=nC, mu_ic=mu_ic, alpha_ic=alpha_ic,
    SL=SL_i, SL_raw=0.85, c_CM=c_CM, o=50., h_d=h_d, b_i=b_i, L_CM=L_CM,
    n_i=n_i, c_raw=50., den_raw=1200., unit_raw=0.5,
    o_raw=30., L_raw=1/52, L_AM=L_AM, Leas=Leas, Cap_3DP=5000., a_i=a_i,
    t_central=0., t_dic=t_dic, xi=2.0, seed=seed)

# ── RUN A: SENZA costo backorder-LT (modello originale) ──────────────────
print("="*72)
print("RUN A — SENZA costo backorder analitico da lead time (baseline)")
print("="*72)
res_A = solve_stochastic(**kwargs, nS=100, mip_gap=0.005, time_limit=300,
                         include_backorder_LT=False, verbose=True,
                         scenario_name="6SKU_no_BO_LT")
print_results(res_A, "SENZA backorder-LT", nI, nD, c_CM, c_AM, b_i, L_CM, L_AM)

# ── RUN B: CON costo backorder-LT (modello corretto) ──────────────────────
print("\n"+"="*72)
print("RUN B — CON costo backorder analitico da lead time (corretto)")
print("="*72)
res_B = solve_stochastic(**kwargs, nS=100, mip_gap=0.005, time_limit=300,
                         include_backorder_LT=True, verbose=True,
                         scenario_name="6SKU_with_BO_LT")
print_results(res_B, "CON backorder-LT", nI, nD, c_CM, c_AM, b_i, L_CM, L_AM)

# ── CONFRONTO ──────────────────────────────────────────────────────────────
print("\n"+"="*72)
print("CONFRONTO A (senza) vs B (con costo backorder-LT)")
print("="*72)
am_A = [i for i in range(nI) if res_A['am'][i]] if res_A['am'] is not None else []
am_B = [i for i in range(nI) if res_B['am'][i]] if res_B['am'] is not None else []
print(f"\n  {'':>20} {'AM':>15} {'Costo':>14} {'B medio':>10}")
print(f"  {'-'*62}")
print(f"  {'A - senza BO-LT':>20} {str(am_A):>15} {res_A['obj_real']:>14,.0f} "
      f"{res_A['B'].mean():>10.2f}")
print(f"  {'B - con BO-LT':>20} {str(am_B):>15} {res_B['obj_real']:>14,.0f} "
      f"{res_B['B'].mean():>10.2f}")
if am_A != am_B:
    print(f"\n  Il costo backorder-LT ha CAMBIATO la decisione tecnologica:")
    for i in range(nI):
        ta = "AM" if i in am_A else "CM"
        tb = "AM" if i in am_B else "CM"
        if ta != tb:
            print(f"    SKU {i}: A={ta} -> B={tb}")
else:
    print(f"\n  Stessa decisione tecnologica. Il costo backorder-LT ha "
          f"solo ridistribuito il costo totale (+{res_B['obj_real']-res_A['obj_real']:,.0f}€).")

RUN A — SENZA costo backorder analitico da lead time (baseline)
Set parameter Username
Set parameter LicenseID to value 2800035
Academic license - for non-commercial use only - expires 2027-03-30
Set parameter OutputFlag to value 1
Set parameter MIPGap to value 0.005
Set parameter TimeLimit to value 300
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-1165G7 @ 2.80GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  300
MIPGap  0.005

Optimize a model with 18444 rows, 17790 columns and 66393 nonzeros (Min)
Model fingerprint: 0x20f0c25e
Model has 17463 linear objective coefficients
Model has 96 simple general constraints
  96 PWL
Variable types: 17628 continuous, 162 integer (159 binary)
Coefficient statistics:
  Matrix range     [4e-03, 9e+05]
  Objective range  [2e-03, 2e+04]
  Bounds range     [1e+00, 1e+00]

## With capacity constraint of B=3

In [ ]:
#!/usr/bin/env python3
"""
================================================================================
RAMP-ML — 6 SKU "AM vince per lead time" — CON COSTO BACKORDER ANALITICO
================================================================================
"""

import numpy as np
import gurobipy as gp
from gurobipy import GRB
from scipy.stats import poisson
from itertools import combinations
import time


def expected_backorder(mu_L, beta):
    """E[(D_L - beta)+] for D_L ~ Poisson(mu_L). Identica alla versione CG."""
    if mu_L < 1e-9 or beta < 0:
        return mu_L
    if beta == 0:
        return mu_L
    return max(mu_L*(1-poisson.cdf(beta-1, mu_L)) - beta*(1-poisson.cdf(beta, mu_L)), 0.0)


def solve_stochastic(
    nI, nD, nC, nS,
    mu_ic, alpha_ic,
    SL, SL_raw,
    c_CM, o, h_d, b_i, L_CM,
    n_i, c_raw, den_raw, unit_raw,
    o_raw, L_raw, L_AM, Leas, Cap_3DP, a_i,
    t_central,
    t_dic=None, B_cap=None, costo_km=100.0,
    coord_DC=None, coord_C=None,
    L_card=None, U_card=None,
    mip_gap=0.01, time_limit=600,
    xi=2.0, seed=42,
    warm_start_sol=None,
    verbose=False, scenario_name="default",
    include_backorder_LT=True,   # <-- TOGGLE per confronto A/B
):
    t_start = time.time()
    I, D, C, S = range(nI), range(nD), range(nC), range(nS)
    pi_s = 1.0 / nS
    SL_i = np.full(nI, SL) if np.isscalar(SL) else np.asarray(SL, dtype=float)

    if t_dic is None and coord_DC is not None and coord_C is not None:
        dist = np.array([
            [np.sqrt((coord_DC[d,0]-coord_C[c,0])**2 +
                     (coord_DC[d,1]-coord_C[c,1])**2)
             for c in range(nC)] for d in range(nD)])
        t_dic = np.stack([dist * costo_km] * nI, axis=1)

    # ── Preprocessing ─────────────────────────────────────────────────────
    prod_i  = n_i * c_CM
    vol_i   = (prod_i / 1.30) * 1e-6
    qbar_i  = (vol_i * den_raw) / unit_raw
    ptime_i = prod_i / (1.30 * 0.00525 * 3600)

    def beta_poisson(mu_val, L_val, SL_target):
        lam = mu_val * L_val
        if lam <= 1e-9: return 0
        for s in range(100000):
            if poisson.cdf(s, lam) >= SL_target: return int(s)
        return 99999

    mu_grid_set = {0.0}
    for i in I:
        for r in range(1, nC + 1):
            for combo in combinations([float(mu_ic[i,c]) for c in C], r):
                mu_grid_set.add(round(sum(combo), 6))
    mu_grid = np.array(sorted(mu_grid_set))
    M = len(mu_grid)

    B_CM_grid  = np.zeros((nI, nD, M))
    B_AM_grid  = np.zeros((nI, nD, M))
    C_EOQ_grid = np.zeros((nI, nD, M))
    for i in I:
        for d in D:
            for m, mu_m in enumerate(mu_grid):
                B_CM_grid[i,d,m] = beta_poisson(mu_m, L_CM[i], SL_i[i])
                B_AM_grid[i,d,m] = beta_poisson(mu_m, L_AM[i,d], SL_i[i])
                if mu_m > 0:
                    C_EOQ_grid[i,d,m] = np.sqrt(2*o*h_d[d]*c_CM[i]*mu_m)

    B_bar = B_CM_grid[:,:,-1]
    mu_id_tot = mu_ic.sum(axis=1)
    mu_max_i = np.array([mu_ic[i,:].sum() for i in I])

    contributi_raw = [float(qbar_i[i]*mu_id_tot[i]) for i in I]
    if nI <= 15:
        mu_raw_grid_set = {0.0}
        for r in range(1, nI+1):
            for combo in combinations(contributi_raw, r):
                mu_raw_grid_set.add(round(sum(combo), 4))
        mu_raw_grid = np.array(sorted(mu_raw_grid_set))
    else:
        mu_raw_grid = np.linspace(0, sum(contributi_raw), 200)
    M_RAW = len(mu_raw_grid)
    B_raw_grid = np.zeros((nD, M_RAW))
    C_EOQ_raw_grid = np.zeros((nD, M_RAW))
    B_bar_raw = np.zeros(nD)
    for d in D:
        for m, mu_m in enumerate(mu_raw_grid):
            B_raw_grid[d,m] = beta_poisson(mu_m, L_raw, SL_raw)
            if mu_m > 0:
                C_EOQ_raw_grid[d,m] = np.sqrt(2*o_raw*h_d[d]*c_raw*mu_m)
        B_bar_raw[d] = B_raw_grid[d,-1]

    rng = np.random.default_rng(seed)
    d_ics = rng.poisson(mu_ic[:,:,np.newaxis], size=(nI,nC,nS)).astype(float)

    Q_bar_CM = np.zeros((nI,nD)); F_bar = np.zeros((nI,nD))
    for i in I:
        for d in D:
            Q_bar_CM[i,d] = d_ics[i,:,:].sum(axis=0).max() * xi
            F_bar[i,d] = d_ics[i,:,:].max() * xi
    Q_bar_AM = np.full((nI,nD), Cap_3DP * xi)

    if L_card is None: L_card = np.ones(nI, dtype=int)
    if U_card is None: U_card = np.full(nI, nD, dtype=int)

    # ── PWL breakpoints (comuni a beta_aux e ai nuovi bo_aux) ─────────────
    N_PWL = 30
    pwl_mu_pts = {}
    for i in I:
        pwl_mu_pts[i] = np.linspace(0.0, max(mu_max_i[i], 0.01), N_PWL)

    # ── Modello ───────────────────────────────────────────────────────────
    mdl = gp.Model(f"stoc_{scenario_name}")
    mdl.setParam("OutputFlag", 1 if verbose else 0)
    mdl.setParam("MIPGap", mip_gap)
    mdl.setParam("TimeLimit", time_limit)

    am   = mdl.addVars(nI, vtype=GRB.BINARY, name="am")
    cm   = mdl.addVars(nI, vtype=GRB.BINARY, name="cm")
    y    = mdl.addVars(nD, nI, nC, vtype=GRB.BINARY, name="y")
    dc   = mdl.addVars(nD, nI, vtype=GRB.BINARY, name="dc")
    v    = mdl.addVars(nI, nD, vtype=GRB.BINARY, name="v")
    w    = mdl.addVars(nI, nD, vtype=GRB.BINARY, name="w")
    B    = mdl.addVars(nI, nD, lb=0.0, name="B")
    n3DP = mdl.addVars(nD, vtype=GRB.INTEGER, lb=0, name="n3DP")
    mu_var = mdl.addVars(nD, nI, lb=0.0, name="mu_var")
    C_EOQ_var = mdl.addVars(nD, nI, lb=0.0, name="C_EOQ_var")
    dcAM = mdl.addVars(nD, vtype=GRB.BINARY, name="dcAM")
    C_EOQ_raw_var = mdl.addVars(nD, lb=0.0, name="C_EOQ_raw_var")
    B_raw = mdl.addVars(nD, lb=0.0, name="B_raw")
    mu_raw_contrib = mdl.addVars(nD, nI, lb=0.0, name="mu_raw_contrib")
    mu_raw_total = mdl.addVars(nD, lb=0.0, name="mu_raw_total")
    beta_cm_aux = mdl.addVars(nI, nD, lb=0.0, name="beta_cm_aux")
    beta_am_aux = mdl.addVars(nI, nD, lb=0.0, name="beta_am_aux")

    # ══════════════════════════════════════════════════════════════════════
    # [NUOVO] Variabili ausiliarie PWL per il costo di backorder analitico
    # ══════════════════════════════════════════════════════════════════════
    bo_cm_aux = mdl.addVars(nI, nD, lb=0.0, name="bo_cm_aux")
    bo_am_aux = mdl.addVars(nI, nD, lb=0.0, name="bo_am_aux")

    q_CM  = mdl.addVars(nI, nD, nS, lb=0.0, name="q_CM")
    q_AM  = mdl.addVars(nI, nD, nS, lb=0.0, name="q_AM")
    f     = mdl.addVars(nI, nD, nC, nS, lb=0.0, name="f")
    u     = mdl.addVars(nI, nC, nS, lb=0.0, name="u")
    I_inv = mdl.addVars(nI, nD, nS, lb=0.0, name="I_inv")

    # ── Vincoli primo stadio ──────────────────────────────────────────────
    mdl.addConstrs((am[i]+cm[i]==1 for i in I), name="A1")
    mdl.addConstrs((y[d,i,c]<=dc[d,i] for d in D for i in I for c in C), name="B1")
    mdl.addConstrs((dc[d,i]<=gp.quicksum(y[d,i,c] for c in C) for d in D for i in I), name="B2")
    mdl.addConstrs((gp.quicksum(y[d,i,c] for d in D)==1 for i in I for c in C), name="assign")
    mdl.addConstrs((gp.quicksum(dc[d,i] for d in D)>=L_card[i] for i in I), name="B3_lo")
    mdl.addConstrs((gp.quicksum(dc[d,i] for d in D)<=U_card[i] for i in I), name="B3_up")
    mdl.addConstrs((v[i,d]<=am[i] for i in I for d in D), name="D1")
    mdl.addConstrs((v[i,d]<=dc[d,i] for i in I for d in D), name="D2")
    mdl.addConstrs((v[i,d]>=am[i]+dc[d,i]-1 for i in I for d in D), name="D2b")
    mdl.addConstrs((w[i,d]<=cm[i] for i in I for d in D), name="W1")
    mdl.addConstrs((w[i,d]<=dc[d,i] for i in I for d in D), name="W2")
    mdl.addConstrs((w[i,d]>=cm[i]+dc[d,i]-1 for i in I for d in D), name="W3")

    mdl.addConstrs((mu_var[d,i]==gp.quicksum(mu_ic[i,c]*y[d,i,c] for c in C)
                    for d in D for i in I), name="DG2_direct")
    mdl.addConstrs((mu_var[d,i]<=mu_max_i[i]*dc[d,i] for d in D for i in I), name="DG2_dc")

    for i in I:
        pts = pwl_mu_pts[i].tolist()
        for d in D:
            bcm = [float(beta_poisson(mu, L_CM[i], SL_i[i])) for mu in pts]
            bam = [float(beta_poisson(mu, L_AM[i,d], SL_i[i])) for mu in pts]
            eoq = [float(np.sqrt(2*o*h_d[d]*c_CM[i]*mu)) if mu>0 else 0.0 for mu in pts]
            mdl.addGenConstrPWL(mu_var[d,i], beta_cm_aux[i,d], pts, bcm, name=f"PWL_bCM_{d}_{i}")
            mdl.addGenConstrPWL(mu_var[d,i], beta_am_aux[i,d], pts, bam, name=f"PWL_bAM_{d}_{i}")
            mdl.addGenConstrPWL(mu_var[d,i], C_EOQ_var[d,i], pts, eoq, name=f"PWL_EOQ_{d}_{i}")

            bo_cm_pts = [b_i[i] * expected_backorder(pts[m]*L_CM[i], bcm[m])
                         for m in range(len(pts))]
            bo_am_pts = [b_i[i] * expected_backorder(pts[m]*L_AM[i,d], bam[m])
                         for m in range(len(pts))]
            mdl.addGenConstrPWL(mu_var[d,i], bo_cm_aux[i,d], pts, bo_cm_pts,
                                name=f"PWL_boCM_{d}_{i}")
            mdl.addGenConstrPWL(mu_var[d,i], bo_am_aux[i,d], pts, bo_am_pts,
                                name=f"PWL_boAM_{d}_{i}")

    mdl.addConstrs((B[i,d]>=beta_cm_aux[i,d]-B_bar[i,d]*(1-cm[i]) for i in I for d in D), name="DG4a")
    mdl.addConstrs((B[i,d]>=beta_am_aux[i,d]-B_bar[i,d]*(1-am[i]) for i in I for d in D), name="DG4b")
    mdl.addConstrs((B[i,d]<=B_bar[i,d]*dc[d,i] for i in I for d in D), name="DG5")

    if B_cap is not None:
        mdl.addConstrs((gp.quicksum(B[i,d] for i in I)<=B_cap for d in D), name="DG5b")

    for d in D:
        for i in I: mdl.addConstr(dcAM[d]>=v[i,d], name=f"dcAM_lo_{d}_{i}")
        mdl.addConstr(dcAM[d]<=gp.quicksum(v[i,d] for i in I), name=f"dcAM_up_{d}")

    M_raw_big = float(max(qbar_i)*sum(mu_ic[i,:].sum() for i in I))*2
    for d in D:
        for i in I:
            mdl.addConstr(mu_raw_contrib[d,i]<=M_raw_big*v[i,d], name=f"RC1_{d}_{i}")
            mdl.addConstr(mu_raw_contrib[d,i]<=qbar_i[i]*mu_var[d,i], name=f"RC2_{d}_{i}")
            mdl.addConstr(mu_raw_contrib[d,i]>=qbar_i[i]*mu_var[d,i]-M_raw_big*(1-v[i,d]), name=f"RC3_{d}_{i}")
        mdl.addConstr(mu_raw_total[d]==gp.quicksum(mu_raw_contrib[d,i] for i in I), name=f"mu_raw_tot_{d}")
        mdl.addConstr(mu_raw_total[d]<=M_raw_big*dcAM[d], name=f"mu_raw_dc_{d}")

    mu_raw_max = float(sum(qbar_i[i]*mu_ic[i,:].sum() for i in I))
    mu_raw_pts = np.linspace(0.0, max(mu_raw_max,0.01), 30).tolist()
    for d in D:
        brp = [float(beta_poisson(mu, L_raw, SL_raw)) for mu in mu_raw_pts]
        erp = [float(np.sqrt(2*o_raw*h_d[d]*c_raw*mu)) if mu>0 else 0.0 for mu in mu_raw_pts]
        bra = mdl.addVar(lb=0.0, name=f"b_raw_aux_{d}")
        mdl.addGenConstrPWL(mu_raw_total[d], bra, mu_raw_pts, brp, name=f"PWL_Braw_{d}")
        mdl.addConstr(B_raw[d]>=bra, name=f"ER4_{d}")
        mdl.addConstr(B_raw[d]<=B_bar_raw[d]*dcAM[d], name=f"ER5_{d}")
        mdl.addGenConstrPWL(mu_raw_total[d], C_EOQ_raw_var[d], mu_raw_pts, erp, name=f"PWL_EOQraw_{d}")

    M_EOQ_big = float(C_EOQ_grid.max())*2
    C_EOQ_act = mdl.addVars(nD, nI, lb=0.0, name="C_EOQ_act")
    for d in D:
        for i in I:
            mdl.addConstr(C_EOQ_act[d,i]<=M_EOQ_big*w[i,d], name=f"L_EOQ1_{d}_{i}")
            mdl.addConstr(C_EOQ_act[d,i]<=C_EOQ_var[d,i], name=f"L_EOQ2_{d}_{i}")
            mdl.addConstr(C_EOQ_act[d,i]>=C_EOQ_var[d,i]-M_EOQ_big*(1-w[i,d]), name=f"L_EOQ3_{d}_{i}")

    M_BO_big = max(
        max(b_i[i] * mu_max_i[i] for i in I),
        1.0
    ) * 2
    BO_act = mdl.addVars(nD, nI, lb=0.0, name="BO_act")
    for d in D:
        for i in I:
            mdl.addConstr(BO_act[d,i] <= M_BO_big*cm[i] + M_BO_big*am[i], name=f"BO_cap_{d}_{i}")
            mdl.addConstr(BO_act[d,i] >= bo_cm_aux[i,d] - M_BO_big*(1-cm[i]), name=f"BO_lo_CM_{d}_{i}")
            mdl.addConstr(BO_act[d,i] >= bo_am_aux[i,d] - M_BO_big*(1-am[i]), name=f"BO_lo_AM_{d}_{i}")
            mdl.addConstr(BO_act[d,i] <= bo_cm_aux[i,d] + M_BO_big*(1-cm[i]), name=f"BO_up_CM_{d}_{i}")
            mdl.addConstr(BO_act[d,i] <= bo_am_aux[i,d] + M_BO_big*(1-am[i]), name=f"BO_up_AM_{d}_{i}")
            mdl.addConstr(BO_act[d,i] <= M_BO_big*dc[d,i], name=f"BO_dc_{d}_{i}")

    ratio_pt = np.where(qbar_i>1e-9, ptime_i/qbar_i, 0.0)
    mdl.addConstrs((n3DP[d]>=gp.quicksum(ratio_pt[i]*mu_raw_contrib[d,i] for i in I)/Cap_3DP for d in D), name="E1")

    # ══════════════════════════════════════════════════════════════════════
    # McCormick per tecnologia-specifico Base Stock
    # ══════════════════════════════════════════════════════════════════════
    B_cm_lin = mdl.addVars(nI, nD, lb=0.0, name="B_cm_lin")
    B_am_lin = mdl.addVars(nI, nD, lb=0.0, name="B_am_lin")
    for i in I:
        for d in D:
            Bub = float(B_bar[i,d])
            mdl.addConstr(B_cm_lin[i,d] <= Bub * w[i,d])
            mdl.addConstr(B_cm_lin[i,d] <= B[i,d])
            mdl.addConstr(B_cm_lin[i,d] >= B[i,d] - Bub*(1-w[i,d]))
            mdl.addConstr(B_am_lin[i,d] <= Bub * v[i,d])
            mdl.addConstr(B_am_lin[i,d] <= B[i,d])
            mdl.addConstr(B_am_lin[i,d] >= B[i,d] - Bub*(1-v[i,d]))

    cost_BS = gp.quicksum(c_CM[i]*(1+h_d[d])*B_cm_lin[i,d] + prod_i[i]*(1+h_d[d])*B_am_lin[i,d] for i in I for d in D)
    cost_EOQ = gp.quicksum(C_EOQ_act[d,i] for i in I for d in D)
    # cost_raw RIMOSSO: consumo diretto materia prima gia' incluso in prod_i,
    # contarlo separatamente era doppio conteggio. Restano EOQ_raw/holding_raw.
    cost_raw_dc = gp.quicksum(C_EOQ_raw_var[d]+h_d[d]*c_raw*B_raw[d] for d in D)
    cost_leas = gp.quicksum(Leas*n3DP[d] for d in D)

    if include_backorder_LT:
        cost_backorder_LT = gp.quicksum(BO_act[d,i] for i in I for d in D)
    else:
        cost_backorder_LT = 0.0

    # Secondo stadio (invariato)
    mdl.addConstrs((gp.quicksum(a_i[i]*q_AM[i,d,s] for i in I)<=Cap_3DP*n3DP[d] for d in D for s in S), name="E2")
    mdl.addConstrs((B[i,d]+q_CM[i,d,s]+q_AM[i,d,s]-gp.quicksum(f[i,d,c,s] for c in C)==I_inv[i,d,s]
                    for i in I for d in D for s in S), name="F1")
    mdl.addConstrs((gp.quicksum(f[i,d,c,s] for d in D)+u[i,c,s]==d_ics[i,c,s]
                    for i in I for c in C for s in S), name="F2")
    mdl.addConstrs((f[i,d,c,s]<=F_bar[i,d]*dc[d,i] for i in I for d in D for c in C for s in S), name="G1")
    mdl.addConstrs((q_CM[i,d,s]<=Q_bar_CM[i,d]*w[i,d] for i in I for d in D for s in S), name="G2")
    mdl.addConstrs((q_AM[i,d,s]<=Q_bar_AM[i,d]*v[i,d] for i in I for d in D for s in S), name="G3")
    mdl.addConstrs((gp.quicksum(pi_s*gp.quicksum(f[i,d,c,s] for d in D) for s in S)
                    >=alpha_ic[i,c]*gp.quicksum(pi_s*d_ics[i,c,s] for s in S)
                    for i in I for c in C), name="H")

    cost_2nd = gp.quicksum(pi_s*(
        gp.quicksum(c_CM[i]*q_CM[i,d,s] for i in I for d in D)+
        gp.quicksum(prod_i[i]*q_AM[i,d,s] for i in I for d in D)+
        gp.quicksum(t_dic[d,i,c]*f[i,d,c,s] for i in I for d in D for c in C)+
        gp.quicksum(h_d[d]*I_inv[i,d,s] for i in I for d in D)+
        gp.quicksum(b_i[i]*u[i,c,s] for i in I for c in C)
    ) for s in S)

    mdl.setObjective(cost_BS+cost_EOQ+cost_raw_dc+cost_leas
                     +cost_backorder_LT+cost_2nd, GRB.MINIMIZE)

    # VarHint
    from scipy.stats import poisson as _pois
    def _bp(mu, L, sl):
        lam = mu*L
        if lam<1e-9: return 0
        for ss in range(10000):
            if _pois.cdf(ss, lam)>=sl: return ss
        return 9999
    for i in I:
        mu_c = float(mu_ic[i,:].sum())
        b_cm_h, b_am_h = _bp(mu_c, L_CM[i], SL_i[i]), _bp(mu_c, L_AM[i,0], SL_i[i])
        risp = c_CM[i]*(1+h_d[0])*(b_cm_h-b_am_h)
        bo_cm = b_i[i]*expected_backorder(mu_c*L_CM[i], b_cm_h)
        bo_am = b_i[i]*expected_backorder(mu_c*L_AM[i,0], b_am_h)
        risp += (bo_cm - bo_am) if include_backorder_LT else 0.0
        costo = (prod_i[i]-c_CM[i])*mu_c
        netto = risp - costo
        if netto>0:
            am[i].VarHintVal=1.0; am[i].VarHintPri=10; cm[i].VarHintVal=0.0
        elif -Leas<netto<=0:
            am[i].VarHintVal=1.0; am[i].VarHintPri=5; cm[i].VarHintVal=0.0
        else:
            am[i].VarHintVal=0.0; am[i].VarHintPri=1; cm[i].VarHintVal=1.0
    mdl.update()

    mdl.optimize()
    solve_time = time.time() - t_start

    if mdl.Status not in [GRB.OPTIMAL, GRB.TIME_LIMIT, GRB.SUBOPTIMAL]:
        return {'status':'INFEASIBLE','obj':None,'obj_real':None,'gap':None,
                'time':solve_time,'am':None,'cm':None,'dc':None,'B':None,
                'n3DP':None,'cost_breakdown':None,'y_sol':None}

    status = {GRB.OPTIMAL:'OPTIMAL',GRB.TIME_LIMIT:'TIME_LIMIT',
              GRB.SUBOPTIMAL:'SUBOPTIMAL'}.get(mdl.Status, str(mdl.Status))

    def snap(val): return val if val>0.5 else 0.0
    am_sol = np.array([am[i].X>0.5 for i in I])
    cm_sol = np.array([cm[i].X>0.5 for i in I])
    dc_sol = np.array([[dc[d,i].X>0.5 for i in I] for d in D])
    B_sol  = np.array([[B[i,d].X for d in D] for i in I])
    n3DP_sol = np.array([max(0.0, n3DP[d].X) for d in D])
    w_sol = np.array([[w[i,d].X for d in D] for i in I])
    v_sol = np.array([[v[i,d].X for d in D] for i in I])
    y_sol = np.array([[[snap(y[d,i,c].X) for c in C] for i in I] for d in D])
    n_dc_per_sku = dc_sol.sum(axis=0)

    bo_lt_sol = sum(BO_act[d,i].X for i in I for d in D)

    mu_var_sol = np.array([[sum(mu_ic[i,c]*y_sol[d,i,c] for c in C) for i in I] for d in D])
    C_EOQ_act_sol = np.zeros((nD,nI))
    for d in D:
        for i in I:
            if w_sol[i,d]>0.5:
                m = int(np.argmin(np.abs(mu_grid-mu_var_sol[d,i])))
                C_EOQ_act_sol[d,i] = C_EOQ_grid[i,d,m]
    mu_raw_sol = np.array([[qbar_i[i]*mu_var_sol[d,i]*float(v_sol[i,d]>0.5) for i in I] for d in D])
    dcAM_sol = np.array([any(v_sol[i,d]>0.5 for i in I) for d in D])
    C_EOQ_raw_sol = np.zeros(nD); B_raw_sol_arr = np.zeros(nD)
    for d in D:
        if dcAM_sol[d]:
            mu_raw_d = mu_raw_sol[d,:].sum()
            m = int(np.argmin(np.abs(mu_raw_grid-mu_raw_d)))
            C_EOQ_raw_sol[d] = C_EOQ_raw_grid[d,m]; B_raw_sol_arr[d] = B_raw_grid[d,m]

    cost_BS_sol = sum(
        c_CM[i]*(1+h_d[d])*B_sol[i,d] if w_sol[i,d] > 0.5
        else (prod_i[i]*(1+h_d[d])*B_sol[i,d] if v_sol[i,d] > 0.5 else 0.0)
        for i in I for d in D
    )
    cost_EOQ_sol = sum(C_EOQ_act_sol[d,i] for d in D for i in I))
    cost_raw_dc_sol = sum(C_EOQ_raw_sol[d]+h_d[d]*c_raw*B_raw_sol_arr[d] for d in D)
    cost_leas_sol = sum(Leas*n3DP_sol[d] for d in D)
    cost_1st = cost_BS_sol+cost_EOQ_sol+cost_raw_dc_sol+cost_leas_sol+bo_lt_sol

    pp = gp.Model("pp"); pp.setParam("OutputFlag",0)
    qp = pp.addVars(nI,nD,nS,lb=0.0); qa = pp.addVars(nI,nD,nS,lb=0.0)
    fp = pp.addVars(nI,nD,nC,nS,lb=0.0); up = pp.addVars(nI,nC,nS,lb=0.0)
    ip = pp.addVars(nI,nD,nS,lb=0.0)
    pp.addConstrs((gp.quicksum(a_i[i]*qa[i,d,s] for i in I)<=Cap_3DP*n3DP_sol[d] for d in D for s in S))
    pp.addConstrs((B_sol[i,d]+qp[i,d,s]+qa[i,d,s]-gp.quicksum(fp[i,d,c,s] for c in C)==ip[i,d,s]
                   for i in I for d in D for s in S))
    pp.addConstrs((gp.quicksum(fp[i,d,c,s] for d in D)+up[i,c,s]==d_ics[i,c,s]
                   for i in I for c in C for s in S))
    pp.addConstrs((fp[i,d,c,s]<=F_bar[i,d]*float(dc_sol[d,i]) for i in I for d in D for c in C for s in S))
    pp.addConstrs((qp[i,d,s]<=Q_bar_CM[i,d]*float(w_sol[i,d]>0.5) for i in I for d in D for s in S))
    pp.addConstrs((qa[i,d,s]<=Q_bar_AM[i,d]*float(v_sol[i,d]>0.5) for i in I for d in D for s in S))
    pp.setObjective(gp.quicksum(pi_s*(
        gp.quicksum(c_CM[i]*qp[i,d,s] for i in I for d in D)+
        gp.quicksum(prod_i[i]*qa[i,d,s] for i in I for d in D)+
        gp.quicksum(t_dic[d,i,c]*fp[i,d,c,s] for i in I for d in D for c in C)+
        gp.quicksum(h_d[d]*ip[i,d,s] for i in I for d in D)+
        gp.quicksum(b_i[i]*up[i,c,s] for i in I for c in C)) for s in S), GRB.MINIMIZE)
    pp.optimize()

    if pp.Status==GRB.OPTIMAL:
        cost_breakdown = {
            'base_stock': cost_BS_sol, 'EOQ_CM': cost_EOQ_sol,
            'EOQ_raw': sum(C_EOQ_raw_sol), 'holding_raw': sum(h_d[d]*c_raw*B_raw_sol_arr[d] for d in D),
            'leasing_3DP': cost_leas_sol,
            'backorder_LT_analitico': bo_lt_sol,
            'trasporto': sum(pi_s*t_dic[d,i,c]*fp[i,d,c,s].X for i in I for d in D for c in C for s in S),
            'produzione_CM': sum(pi_s*c_CM[i]*qp[i,d,s].X for i in I for d in D for s in S),
            'produzione_AM': sum(pi_s*prod_i[i]*qa[i,d,s].X for i in I for d in D for s in S),
            'holding_inv': sum(pi_s*h_d[d]*ip[i,d,s].X for i in I for d in D for s in S),
            'backorder_2ndstage': sum(pi_s*b_i[i]*up[i,c,s].X for i in I for c in C for s in S),
        }
        obj_real = cost_1st+pp.ObjVal
        if abs(obj_real - mdl.ObjVal) > 1.0:
            print(f"  [ATTENZIONE] obj_real ({obj_real:,.2f}) != mdl.ObjVal "
                  f"({mdl.ObjVal:,.2f}), diff={obj_real-mdl.ObjVal:,.2f}€")
    else:
        cost_breakdown = {}
        obj_real = mdl.ObjVal

    return {'status':status,'obj':mdl.ObjVal,'obj_real':obj_real,
            'gap':mdl.MIPGap*100,'time':solve_time,
            'am':am_sol,'cm':cm_sol,'dc':dc_sol,'B':B_sol,
            'n3DP':n3DP_sol,'n_dc_per_sku':n_dc_per_sku,
            'cost_breakdown':cost_breakdown,'y_sol':y_sol}


def print_results(res, name, nI, nD, c_CM, c_AM, b_i, L_CM, L_AM):
    print(f"\n{'='*70}\nRISULTATI: {name}\n{'='*70}")
    if res['status']=='INFEASIBLE': print("  INFEASIBLE"); return
    print(f"  Status: {res['status']}  |  Costo: {res['obj_real']:,.2f}€  |  "
          f"Gap: {res['gap']:.2f}%  |  Tempo: {res['time']:.1f}s")
    print(f"\n  {'SKU':<4} {'T':<3} {'DC':<10} {'B':>4} {'c_CM':>7} {'c_AM':>7} {'ratio':>6}")
    print(f"  {'-'*55}")
    for i in range(nI):
        t = "AM" if res['am'][i] else "CM"
        dcs = [d for d in range(nD) if res['dc'][d,i]]
        bm = res['B'][i,dcs].mean() if dcs else 0
        print(f"  {i:<4} {t:<3} {str(dcs):<10} {bm:>4.1f} {c_CM[i]:>7.0f} "
              f"{c_AM[i]:>7.0f} {c_AM[i]/c_CM[i]:>6.2f}")
    print(f"\n  Stampanti: {np.round(res['n3DP'],1)}")
    if res.get('cost_breakdown'):
        print(f"  Costi:")
        tot = 0.0
        for k,v in res['cost_breakdown'].items():
            if abs(v)>0.01:
                print(f"    {k:<24}: {v:>12.2f}€")
                tot += v
        print(f"    {'TOTALE':<24}: {tot:>12.2f}€")


# ══════════════════════════════════════════════════════════════════════════
# CASO 6 SKU — "AM vince per lead time, non per prezzo"
# ══════════════════════════════════════════════════════════════════════════
nI, nD, nC = 6, 3, 5
seed = 42

coord_DC = np.array([[1.0,8.0],[1.0,2.0],[4.0,5.0]])
coord_C  = np.array([[1.5,9.0],[0.5,7.5],[1.5,1.0],[0.5,2.5],[4.5,5.5]])

c_CM = np.array([5000.,5000., 1000.,1000., 200.,200.])
c_AM = np.array([6000.,6000., 1500.,1500., 800.,800.])
n_i  = c_AM / c_CM

L_CM = np.array([26,26, 26,26, 9,9]) / 52
L_AM_vec = np.array([1,1, 1,1, 2,2]) / 52
L_AM = np.column_stack([L_AM_vec] * nD)

b_i  = np.array([150000.,120000., 50000.,40000., 3000.,2000.])
SL_i = np.array([0.99,0.99, 0.99,0.99, 0.90,0.88])

mu_ic = np.array([
    [0.8,0.7,0.5,0.5,0.5],
    [0.6,0.6,0.6,0.6,0.6],
    [0.5,0.4,0.4,0.4,0.3],
    [0.4,0.4,0.4,0.4,0.4],
    [2.0,2.0,2.0,2.0,2.0],
    [2.0,2.0,2.0,2.0,2.0],
])
alpha_ic = np.column_stack([SL_i] * nC)
a_i = c_AM / (1.30 * 0.00525 * 3600)
h_d = np.full(nD, 0.20)

costo_km = 50.0
Leas     = 15000.0

dist = np.array([[np.sqrt((coord_DC[d,0]-coord_C[c,0])**2 +
                          (coord_DC[d,1]-coord_C[c,1])**2)
                  for c in range(nC)] for d in range(nD)])
t_dic = np.stack([dist * costo_km] * nI, axis=1)

kwargs = dict(nI=nI, nD=nD, nC=nC, mu_ic=mu_ic, alpha_ic=alpha_ic,
    SL=SL_i, SL_raw=0.85, c_CM=c_CM, o=50., h_d=h_d, b_i=b_i, L_CM=L_CM,
    n_i=n_i, c_raw=50., den_raw=1200., unit_raw=0.5,
    o_raw=30., L_raw=1/52, L_AM=L_AM, Leas=Leas, Cap_3DP=5000., a_i=a_i,
    t_central=0., t_dic=t_dic, xi=2.0, seed=seed)

# ── RUN A: SENZA costo backorder-LT (modello originale) ──────────────────
print("="*72)
print("RUN A — SENZA costo backorder analitico da lead time (baseline)")
print("="*72)
res_A = solve_stochastic(**kwargs, nS=100, mip_gap=0.005, time_limit=300,
                         include_backorder_LT=False, verbose=True,B_cap=3,
                         scenario_name="6SKU_no_BO_LT")
print_results(res_A, "SENZA backorder-LT", nI, nD, c_CM, c_AM, b_i, L_CM, L_AM)

# ── RUN B: CON costo backorder-LT (modello corretto) ──────────────────────
print("\n"+"="*72)
print("RUN B — CON costo backorder analitico da lead time (corretto)")
print("="*72)
res_B = solve_stochastic(**kwargs, nS=100, mip_gap=0.005, time_limit=300,
                         include_backorder_LT=True, verbose=True,B_cap=3,
                         scenario_name="6SKU_with_BO_LT")
print_results(res_B, "CON backorder-LT", nI, nD, c_CM, c_AM, b_i, L_CM, L_AM)

# ── CONFRONTO ──────────────────────────────────────────────────────────────
print("\n"+"="*72)
print("CONFRONTO A (senza) vs B (con costo backorder-LT)")
print("="*72)
am_A = [i for i in range(nI) if res_A['am'][i]] if res_A['am'] is not None else []
am_B = [i for i in range(nI) if res_B['am'][i]] if res_B['am'] is not None else []
print(f"\n  {'':>20} {'AM':>15} {'Costo':>14} {'B medio':>10}")
print(f"  {'-'*62}")
print(f"  {'A - senza BO-LT':>20} {str(am_A):>15} {res_A['obj_real']:>14,.0f} "
      f"{res_A['B'].mean():>10.2f}")
print(f"  {'B - con BO-LT':>20} {str(am_B):>15} {res_B['obj_real']:>14,.0f} "
      f"{res_B['B'].mean():>10.2f}")
if am_A != am_B:
    print(f"\n  Il costo backorder-LT ha CAMBIATO la decisione tecnologica:")
    for i in range(nI):
        ta = "AM" if i in am_A else "CM"
        tb = "AM" if i in am_B else "CM"
        if ta != tb:
            print(f"    SKU {i}: A={ta} -> B={tb}")
else:
    print(f"\n  Stessa decisione tecnologica. Il costo backorder-LT ha "
          f"solo ridistribuito il costo totale (+{res_B['obj_real']-res_A['obj_real']:,.0f}€).")

RUN A — SENZA costo backorder analitico da lead time (baseline)
Set parameter OutputFlag to value 1
Set parameter MIPGap to value 0.005
Set parameter TimeLimit to value 300
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-1165G7 @ 2.80GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  300
MIPGap  0.005

Optimize a model with 18447 rows, 17790 columns and 66411 nonzeros (Min)
Model fingerprint: 0x2886e4a9
Model has 17463 linear objective coefficients
Model has 96 simple general constraints
  96 PWL
Variable types: 17628 continuous, 162 integer (159 binary)
Coefficient statistics:
  Matrix range     [4e-03, 9e+05]
  Objective range  [2e-03, 2e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [3e-01, 9e+05]
  PWLCon x range   [7e-02, 1e+02]
  PWLCon y range   [0e+00, 2e+03]

Using variable hints.
Presol

## Scenarios stability

30S

In [ ]:
#!/usr/bin/env python3
"""
================================================================================
RAMP-ML — 6 SKU "AM vince per lead time" — CON COSTO BACKORDER ANALITICO
================================================================================
"""

import numpy as np
import gurobipy as gp
from gurobipy import GRB
from scipy.stats import poisson
from itertools import combinations
import time


def expected_backorder(mu_L, beta):
    """E[(D_L - beta)+] for D_L ~ Poisson(mu_L). Identica alla versione CG."""
    if mu_L < 1e-9 or beta < 0:
        return mu_L
    if beta == 0:
        return mu_L
    return max(mu_L*(1-poisson.cdf(beta-1, mu_L)) - beta*(1-poisson.cdf(beta, mu_L)), 0.0)


def solve_stochastic(
    nI, nD, nC, nS,
    mu_ic, alpha_ic,
    SL, SL_raw,
    c_CM, o, h_d, b_i, L_CM,
    n_i, c_raw, den_raw, unit_raw,
    o_raw, L_raw, L_AM, Leas, Cap_3DP, a_i,
    t_central,
    t_dic=None, B_cap=None, costo_km=100.0,
    coord_DC=None, coord_C=None,
    L_card=None, U_card=None,
    mip_gap=0.01, time_limit=600,
    xi=2.0, seed=42,
    warm_start_sol=None,
    verbose=False, scenario_name="default",
    include_backorder_LT=True,   # <-- TOGGLE per confronto A/B
):
    t_start = time.time()
    I, D, C, S = range(nI), range(nD), range(nC), range(nS)
    pi_s = 1.0 / nS
    SL_i = np.full(nI, SL) if np.isscalar(SL) else np.asarray(SL, dtype=float)

    if t_dic is None and coord_DC is not None and coord_C is not None:
        dist = np.array([
            [np.sqrt((coord_DC[d,0]-coord_C[c,0])**2 +
                     (coord_DC[d,1]-coord_C[c,1])**2)
             for c in range(nC)] for d in range(nD)])
        t_dic = np.stack([dist * costo_km] * nI, axis=1)

    # ── Preprocessing ─────────────────────────────────────────────────────
    prod_i  = n_i * c_CM
    vol_i   = (prod_i / 1.30) * 1e-6
    qbar_i  = (vol_i * den_raw) / unit_raw
    ptime_i = prod_i / (1.30 * 0.00525 * 3600)

    def beta_poisson(mu_val, L_val, SL_target):
        lam = mu_val * L_val
        if lam <= 1e-9: return 0
        for s in range(100000):
            if poisson.cdf(s, lam) >= SL_target: return int(s)
        return 99999

    mu_grid_set = {0.0}
    for i in I:
        for r in range(1, nC + 1):
            for combo in combinations([float(mu_ic[i,c]) for c in C], r):
                mu_grid_set.add(round(sum(combo), 6))
    mu_grid = np.array(sorted(mu_grid_set))
    M = len(mu_grid)

    B_CM_grid  = np.zeros((nI, nD, M))
    B_AM_grid  = np.zeros((nI, nD, M))
    C_EOQ_grid = np.zeros((nI, nD, M))
    for i in I:
        for d in D:
            for m, mu_m in enumerate(mu_grid):
                B_CM_grid[i,d,m] = beta_poisson(mu_m, L_CM[i], SL_i[i])
                B_AM_grid[i,d,m] = beta_poisson(mu_m, L_AM[i,d], SL_i[i])
                if mu_m > 0:
                    C_EOQ_grid[i,d,m] = np.sqrt(2*o*h_d[d]*c_CM[i]*mu_m)

    B_bar = B_CM_grid[:,:,-1]
    mu_id_tot = mu_ic.sum(axis=1)
    mu_max_i = np.array([mu_ic[i,:].sum() for i in I])

    contributi_raw = [float(qbar_i[i]*mu_id_tot[i]) for i in I]
    if nI <= 15:
        mu_raw_grid_set = {0.0}
        for r in range(1, nI+1):
            for combo in combinations(contributi_raw, r):
                mu_raw_grid_set.add(round(sum(combo), 4))
        mu_raw_grid = np.array(sorted(mu_raw_grid_set))
    else:
        mu_raw_grid = np.linspace(0, sum(contributi_raw), 200)
    M_RAW = len(mu_raw_grid)
    B_raw_grid = np.zeros((nD, M_RAW))
    C_EOQ_raw_grid = np.zeros((nD, M_RAW))
    B_bar_raw = np.zeros(nD)
    for d in D:
        for m, mu_m in enumerate(mu_raw_grid):
            B_raw_grid[d,m] = beta_poisson(mu_m, L_raw, SL_raw)
            if mu_m > 0:
                C_EOQ_raw_grid[d,m] = np.sqrt(2*o_raw*h_d[d]*c_raw*mu_m)
        B_bar_raw[d] = B_raw_grid[d,-1]

    rng = np.random.default_rng(seed)
    d_ics = rng.poisson(mu_ic[:,:,np.newaxis], size=(nI,nC,nS)).astype(float)

    Q_bar_CM = np.zeros((nI,nD)); F_bar = np.zeros((nI,nD))
    for i in I:
        for d in D:
            Q_bar_CM[i,d] = d_ics[i,:,:].sum(axis=0).max() * xi
            F_bar[i,d] = d_ics[i,:,:].max() * xi
    Q_bar_AM = np.full((nI,nD), Cap_3DP * xi)

    if L_card is None: L_card = np.ones(nI, dtype=int)
    if U_card is None: U_card = np.full(nI, nD, dtype=int)

    # ── PWL breakpoints (comuni a beta_aux e ai nuovi bo_aux) ─────────────
    N_PWL = 30
    pwl_mu_pts = {}
    for i in I:
        pwl_mu_pts[i] = np.linspace(0.0, max(mu_max_i[i], 0.01), N_PWL)

    # ── Modello ───────────────────────────────────────────────────────────
    mdl = gp.Model(f"stoc_{scenario_name}")
    mdl.setParam("OutputFlag", 1 if verbose else 0)
    mdl.setParam("MIPGap", mip_gap)
    mdl.setParam("TimeLimit", time_limit)

    am   = mdl.addVars(nI, vtype=GRB.BINARY, name="am")
    cm   = mdl.addVars(nI, vtype=GRB.BINARY, name="cm")
    y    = mdl.addVars(nD, nI, nC, vtype=GRB.BINARY, name="y")
    dc   = mdl.addVars(nD, nI, vtype=GRB.BINARY, name="dc")
    v    = mdl.addVars(nI, nD, vtype=GRB.BINARY, name="v")
    w    = mdl.addVars(nI, nD, vtype=GRB.BINARY, name="w")
    B    = mdl.addVars(nI, nD, lb=0.0, name="B")
    n3DP = mdl.addVars(nD, vtype=GRB.INTEGER, lb=0, name="n3DP")
    mu_var = mdl.addVars(nD, nI, lb=0.0, name="mu_var")
    C_EOQ_var = mdl.addVars(nD, nI, lb=0.0, name="C_EOQ_var")
    dcAM = mdl.addVars(nD, vtype=GRB.BINARY, name="dcAM")
    C_EOQ_raw_var = mdl.addVars(nD, lb=0.0, name="C_EOQ_raw_var")
    B_raw = mdl.addVars(nD, lb=0.0, name="B_raw")
    mu_raw_contrib = mdl.addVars(nD, nI, lb=0.0, name="mu_raw_contrib")
    mu_raw_total = mdl.addVars(nD, lb=0.0, name="mu_raw_total")
    beta_cm_aux = mdl.addVars(nI, nD, lb=0.0, name="beta_cm_aux")
    beta_am_aux = mdl.addVars(nI, nD, lb=0.0, name="beta_am_aux")

    # ══════════════════════════════════════════════════════════════════════
    # [NUOVO] Variabili ausiliarie PWL per il costo di backorder analitico
    # ══════════════════════════════════════════════════════════════════════
    bo_cm_aux = mdl.addVars(nI, nD, lb=0.0, name="bo_cm_aux")
    bo_am_aux = mdl.addVars(nI, nD, lb=0.0, name="bo_am_aux")

    q_CM  = mdl.addVars(nI, nD, nS, lb=0.0, name="q_CM")
    q_AM  = mdl.addVars(nI, nD, nS, lb=0.0, name="q_AM")
    f     = mdl.addVars(nI, nD, nC, nS, lb=0.0, name="f")
    u     = mdl.addVars(nI, nC, nS, lb=0.0, name="u")
    I_inv = mdl.addVars(nI, nD, nS, lb=0.0, name="I_inv")

    # ── Vincoli primo stadio ──────────────────────────────────────────────
    mdl.addConstrs((am[i]+cm[i]==1 for i in I), name="A1")
    mdl.addConstrs((y[d,i,c]<=dc[d,i] for d in D for i in I for c in C), name="B1")
    mdl.addConstrs((dc[d,i]<=gp.quicksum(y[d,i,c] for c in C) for d in D for i in I), name="B2")
    mdl.addConstrs((gp.quicksum(y[d,i,c] for d in D)==1 for i in I for c in C), name="assign")
    mdl.addConstrs((gp.quicksum(dc[d,i] for d in D)>=L_card[i] for i in I), name="B3_lo")
    mdl.addConstrs((gp.quicksum(dc[d,i] for d in D)<=U_card[i] for i in I), name="B3_up")
    mdl.addConstrs((v[i,d]<=am[i] for i in I for d in D), name="D1")
    mdl.addConstrs((v[i,d]<=dc[d,i] for i in I for d in D), name="D2")
    mdl.addConstrs((v[i,d]>=am[i]+dc[d,i]-1 for i in I for d in D), name="D2b")
    mdl.addConstrs((w[i,d]<=cm[i] for i in I for d in D), name="W1")
    mdl.addConstrs((w[i,d]<=dc[d,i] for i in I for d in D), name="W2")
    mdl.addConstrs((w[i,d]>=cm[i]+dc[d,i]-1 for i in I for d in D), name="W3")

    mdl.addConstrs((mu_var[d,i]==gp.quicksum(mu_ic[i,c]*y[d,i,c] for c in C)
                    for d in D for i in I), name="DG2_direct")
    mdl.addConstrs((mu_var[d,i]<=mu_max_i[i]*dc[d,i] for d in D for i in I), name="DG2_dc")

    for i in I:
        pts = pwl_mu_pts[i].tolist()
        for d in D:
            bcm = [float(beta_poisson(mu, L_CM[i], SL_i[i])) for mu in pts]
            bam = [float(beta_poisson(mu, L_AM[i,d], SL_i[i])) for mu in pts]
            eoq = [float(np.sqrt(2*o*h_d[d]*c_CM[i]*mu)) if mu>0 else 0.0 for mu in pts]
            mdl.addGenConstrPWL(mu_var[d,i], beta_cm_aux[i,d], pts, bcm, name=f"PWL_bCM_{d}_{i}")
            mdl.addGenConstrPWL(mu_var[d,i], beta_am_aux[i,d], pts, bam, name=f"PWL_bAM_{d}_{i}")
            mdl.addGenConstrPWL(mu_var[d,i], C_EOQ_var[d,i], pts, eoq, name=f"PWL_EOQ_{d}_{i}")

            bo_cm_pts = [b_i[i] * expected_backorder(pts[m]*L_CM[i], bcm[m])
                         for m in range(len(pts))]
            bo_am_pts = [b_i[i] * expected_backorder(pts[m]*L_AM[i,d], bam[m])
                         for m in range(len(pts))]
            mdl.addGenConstrPWL(mu_var[d,i], bo_cm_aux[i,d], pts, bo_cm_pts,
                                name=f"PWL_boCM_{d}_{i}")
            mdl.addGenConstrPWL(mu_var[d,i], bo_am_aux[i,d], pts, bo_am_pts,
                                name=f"PWL_boAM_{d}_{i}")

    mdl.addConstrs((B[i,d]>=beta_cm_aux[i,d]-B_bar[i,d]*(1-cm[i]) for i in I for d in D), name="DG4a")
    mdl.addConstrs((B[i,d]>=beta_am_aux[i,d]-B_bar[i,d]*(1-am[i]) for i in I for d in D), name="DG4b")
    mdl.addConstrs((B[i,d]<=B_bar[i,d]*dc[d,i] for i in I for d in D), name="DG5")

    if B_cap is not None:
        mdl.addConstrs((gp.quicksum(B[i,d] for i in I)<=B_cap for d in D), name="DG5b")

    for d in D:
        for i in I: mdl.addConstr(dcAM[d]>=v[i,d], name=f"dcAM_lo_{d}_{i}")
        mdl.addConstr(dcAM[d]<=gp.quicksum(v[i,d] for i in I), name=f"dcAM_up_{d}")

    M_raw_big = float(max(qbar_i)*sum(mu_ic[i,:].sum() for i in I))*2
    for d in D:
        for i in I:
            mdl.addConstr(mu_raw_contrib[d,i]<=M_raw_big*v[i,d], name=f"RC1_{d}_{i}")
            mdl.addConstr(mu_raw_contrib[d,i]<=qbar_i[i]*mu_var[d,i], name=f"RC2_{d}_{i}")
            mdl.addConstr(mu_raw_contrib[d,i]>=qbar_i[i]*mu_var[d,i]-M_raw_big*(1-v[i,d]), name=f"RC3_{d}_{i}")
        mdl.addConstr(mu_raw_total[d]==gp.quicksum(mu_raw_contrib[d,i] for i in I), name=f"mu_raw_tot_{d}")
        mdl.addConstr(mu_raw_total[d]<=M_raw_big*dcAM[d], name=f"mu_raw_dc_{d}")

    mu_raw_max = float(sum(qbar_i[i]*mu_ic[i,:].sum() for i in I))
    mu_raw_pts = np.linspace(0.0, max(mu_raw_max,0.01), 30).tolist()
    for d in D:
        brp = [float(beta_poisson(mu, L_raw, SL_raw)) for mu in mu_raw_pts]
        erp = [float(np.sqrt(2*o_raw*h_d[d]*c_raw*mu)) if mu>0 else 0.0 for mu in mu_raw_pts]
        bra = mdl.addVar(lb=0.0, name=f"b_raw_aux_{d}")
        mdl.addGenConstrPWL(mu_raw_total[d], bra, mu_raw_pts, brp, name=f"PWL_Braw_{d}")
        mdl.addConstr(B_raw[d]>=bra, name=f"ER4_{d}")
        mdl.addConstr(B_raw[d]<=B_bar_raw[d]*dcAM[d], name=f"ER5_{d}")
        mdl.addGenConstrPWL(mu_raw_total[d], C_EOQ_raw_var[d], mu_raw_pts, erp, name=f"PWL_EOQraw_{d}")

    M_EOQ_big = float(C_EOQ_grid.max())*2
    C_EOQ_act = mdl.addVars(nD, nI, lb=0.0, name="C_EOQ_act")
    for d in D:
        for i in I:
            mdl.addConstr(C_EOQ_act[d,i]<=M_EOQ_big*w[i,d], name=f"L_EOQ1_{d}_{i}")
            mdl.addConstr(C_EOQ_act[d,i]<=C_EOQ_var[d,i], name=f"L_EOQ2_{d}_{i}")
            mdl.addConstr(C_EOQ_act[d,i]>=C_EOQ_var[d,i]-M_EOQ_big*(1-w[i,d]), name=f"L_EOQ3_{d}_{i}")

    M_BO_big = max(
        max(b_i[i] * mu_max_i[i] for i in I),
        1.0
    ) * 2
    BO_act = mdl.addVars(nD, nI, lb=0.0, name="BO_act")
    for d in D:
        for i in I:
            mdl.addConstr(BO_act[d,i] <= M_BO_big*cm[i] + M_BO_big*am[i], name=f"BO_cap_{d}_{i}")
            mdl.addConstr(BO_act[d,i] >= bo_cm_aux[i,d] - M_BO_big*(1-cm[i]), name=f"BO_lo_CM_{d}_{i}")
            mdl.addConstr(BO_act[d,i] >= bo_am_aux[i,d] - M_BO_big*(1-am[i]), name=f"BO_lo_AM_{d}_{i}")
            mdl.addConstr(BO_act[d,i] <= bo_cm_aux[i,d] + M_BO_big*(1-cm[i]), name=f"BO_up_CM_{d}_{i}")
            mdl.addConstr(BO_act[d,i] <= bo_am_aux[i,d] + M_BO_big*(1-am[i]), name=f"BO_up_AM_{d}_{i}")
            mdl.addConstr(BO_act[d,i] <= M_BO_big*dc[d,i], name=f"BO_dc_{d}_{i}")

    ratio_pt = np.where(qbar_i>1e-9, ptime_i/qbar_i, 0.0)
    mdl.addConstrs((n3DP[d]>=gp.quicksum(ratio_pt[i]*mu_raw_contrib[d,i] for i in I)/Cap_3DP for d in D), name="E1")

    # ══════════════════════════════════════════════════════════════════════
    # McCormick per tecnologia-specifico Base Stock
    # ══════════════════════════════════════════════════════════════════════
    B_cm_lin = mdl.addVars(nI, nD, lb=0.0, name="B_cm_lin")
    B_am_lin = mdl.addVars(nI, nD, lb=0.0, name="B_am_lin")
    for i in I:
        for d in D:
            Bub = float(B_bar[i,d])
            mdl.addConstr(B_cm_lin[i,d] <= Bub * w[i,d])
            mdl.addConstr(B_cm_lin[i,d] <= B[i,d])
            mdl.addConstr(B_cm_lin[i,d] >= B[i,d] - Bub*(1-w[i,d]))
            mdl.addConstr(B_am_lin[i,d] <= Bub * v[i,d])
            mdl.addConstr(B_am_lin[i,d] <= B[i,d])
            mdl.addConstr(B_am_lin[i,d] >= B[i,d] - Bub*(1-v[i,d]))

    cost_BS = gp.quicksum(c_CM[i]*(1+h_d[d])*B_cm_lin[i,d] + prod_i[i]*(1+h_d[d])*B_am_lin[i,d] for i in I for d in D)
    cost_EOQ = gp.quicksum(C_EOQ_act[d,i] for i in I for d in D)
    cost_raw_dc = gp.quicksum(C_EOQ_raw_var[d]+h_d[d]*c_raw*B_raw[d] for d in D)
    cost_leas = gp.quicksum(Leas*n3DP[d] for d in D)

    if include_backorder_LT:
        cost_backorder_LT = gp.quicksum(BO_act[d,i] for i in I for d in D)
    else:
        cost_backorder_LT = 0.0

    # Secondo stadio (invariato)
    mdl.addConstrs((gp.quicksum(a_i[i]*q_AM[i,d,s] for i in I)<=Cap_3DP*n3DP[d] for d in D for s in S), name="E2")
    mdl.addConstrs((B[i,d]+q_CM[i,d,s]+q_AM[i,d,s]-gp.quicksum(f[i,d,c,s] for c in C)==I_inv[i,d,s]
                    for i in I for d in D for s in S), name="F1")
    mdl.addConstrs((gp.quicksum(f[i,d,c,s] for d in D)+u[i,c,s]==d_ics[i,c,s]
                    for i in I for c in C for s in S), name="F2")
    mdl.addConstrs((f[i,d,c,s]<=F_bar[i,d]*dc[d,i] for i in I for d in D for c in C for s in S), name="G1")
    mdl.addConstrs((q_CM[i,d,s]<=Q_bar_CM[i,d]*w[i,d] for i in I for d in D for s in S), name="G2")
    mdl.addConstrs((q_AM[i,d,s]<=Q_bar_AM[i,d]*v[i,d] for i in I for d in D for s in S), name="G3")
    mdl.addConstrs((gp.quicksum(pi_s*gp.quicksum(f[i,d,c,s] for d in D) for s in S)
                    >=alpha_ic[i,c]*gp.quicksum(pi_s*d_ics[i,c,s] for s in S)
                    for i in I for c in C), name="H")

    cost_2nd = gp.quicksum(pi_s*(
        gp.quicksum(c_CM[i]*q_CM[i,d,s] for i in I for d in D)+
        gp.quicksum(prod_i[i]*q_AM[i,d,s] for i in I for d in D)+
        gp.quicksum(t_dic[d,i,c]*f[i,d,c,s] for i in I for d in D for c in C)+
        gp.quicksum(h_d[d]*I_inv[i,d,s] for i in I for d in D)+
        gp.quicksum(b_i[i]*u[i,c,s] for i in I for c in C)
    ) for s in S)

    mdl.setObjective(cost_BS+cost_EOQ+cost_raw_dc+cost_leas
                     +cost_backorder_LT+cost_2nd, GRB.MINIMIZE)

    # VarHint
    from scipy.stats import poisson as _pois
    def _bp(mu, L, sl):
        lam = mu*L
        if lam<1e-9: return 0
        for ss in range(10000):
            if _pois.cdf(ss, lam)>=sl: return ss
        return 9999
    for i in I:
        mu_c = float(mu_ic[i,:].sum())
        b_cm_h, b_am_h = _bp(mu_c, L_CM[i], SL_i[i]), _bp(mu_c, L_AM[i,0], SL_i[i])
        risp = c_CM[i]*(1+h_d[0])*(b_cm_h-b_am_h)
        bo_cm = b_i[i]*expected_backorder(mu_c*L_CM[i], b_cm_h)
        bo_am = b_i[i]*expected_backorder(mu_c*L_AM[i,0], b_am_h)
        risp += (bo_cm - bo_am) if include_backorder_LT else 0.0
        costo = (prod_i[i]-c_CM[i])*mu_c
        netto = risp - costo
        if netto>0:
            am[i].VarHintVal=1.0; am[i].VarHintPri=10; cm[i].VarHintVal=0.0
        elif -Leas<netto<=0:
            am[i].VarHintVal=1.0; am[i].VarHintPri=5; cm[i].VarHintVal=0.0
        else:
            am[i].VarHintVal=0.0; am[i].VarHintPri=1; cm[i].VarHintVal=1.0
    mdl.update()

    mdl.optimize()
    solve_time = time.time() - t_start

    if mdl.Status not in [GRB.OPTIMAL, GRB.TIME_LIMIT, GRB.SUBOPTIMAL]:
        return {'status':'INFEASIBLE','obj':None,'obj_real':None,'gap':None,
                'time':solve_time,'am':None,'cm':None,'dc':None,'B':None,
                'n3DP':None,'cost_breakdown':None,'y_sol':None}

    status = {GRB.OPTIMAL:'OPTIMAL',GRB.TIME_LIMIT:'TIME_LIMIT',
              GRB.SUBOPTIMAL:'SUBOPTIMAL'}.get(mdl.Status, str(mdl.Status))

    def snap(val): return val if val>0.5 else 0.0
    am_sol = np.array([am[i].X>0.5 for i in I])
    cm_sol = np.array([cm[i].X>0.5 for i in I])
    dc_sol = np.array([[dc[d,i].X>0.5 for i in I] for d in D])
    B_sol  = np.array([[B[i,d].X for d in D] for i in I])
    n3DP_sol = np.array([max(0.0, n3DP[d].X) for d in D])
    w_sol = np.array([[w[i,d].X for d in D] for i in I])
    v_sol = np.array([[v[i,d].X for d in D] for i in I])
    y_sol = np.array([[[snap(y[d,i,c].X) for c in C] for i in I] for d in D])
    n_dc_per_sku = dc_sol.sum(axis=0)

    bo_lt_sol = sum(BO_act[d,i].X for i in I for d in D)

    mu_var_sol = np.array([[sum(mu_ic[i,c]*y_sol[d,i,c] for c in C) for i in I] for d in D])
    C_EOQ_act_sol = np.zeros((nD,nI))
    for d in D:
        for i in I:
            if w_sol[i,d]>0.5:
                m = int(np.argmin(np.abs(mu_grid-mu_var_sol[d,i])))
                C_EOQ_act_sol[d,i] = C_EOQ_grid[i,d,m]
    mu_raw_sol = np.array([[qbar_i[i]*mu_var_sol[d,i]*float(v_sol[i,d]>0.5) for i in I] for d in D])
    dcAM_sol = np.array([any(v_sol[i,d]>0.5 for i in I) for d in D])
    C_EOQ_raw_sol = np.zeros(nD); B_raw_sol_arr = np.zeros(nD)
    for d in D:
        if dcAM_sol[d]:
            mu_raw_d = mu_raw_sol[d,:].sum()
            m = int(np.argmin(np.abs(mu_raw_grid-mu_raw_d)))
            C_EOQ_raw_sol[d] = C_EOQ_raw_grid[d,m]; B_raw_sol_arr[d] = B_raw_grid[d,m]

    # FIX: base-stock tecnologia-specifico anche nel reporting (prima usava
    # sempre c_CM anche per gli SKU AM, disallineato dal vero obiettivo
    # che usa B_cm_lin/B_am_lin). w_sol/v_sol selezionano il ramo corretto.
    cost_BS_sol = sum(
        c_CM[i]*(1+h_d[d])*B_sol[i,d] if w_sol[i,d] > 0.5
        else (prod_i[i]*(1+h_d[d])*B_sol[i,d] if v_sol[i,d] > 0.5 else 0.0)
        for i in I for d in D
    )
    cost_EOQ_sol = sum(C_EOQ_act_sol[d,i] for d in D for i in I)
    # cost_raw_sol RIMOSSO (stesso motivo del fix nell'obiettivo)
    cost_raw_dc_sol = sum(C_EOQ_raw_sol[d]+h_d[d]*c_raw*B_raw_sol_arr[d] for d in D)
    cost_leas_sol = sum(Leas*n3DP_sol[d] for d in D)
    cost_1st = cost_BS_sol+cost_EOQ_sol+cost_raw_dc_sol+cost_leas_sol+bo_lt_sol

    pp = gp.Model("pp"); pp.setParam("OutputFlag",0)
    qp = pp.addVars(nI,nD,nS,lb=0.0); qa = pp.addVars(nI,nD,nS,lb=0.0)
    fp = pp.addVars(nI,nD,nC,nS,lb=0.0); up = pp.addVars(nI,nC,nS,lb=0.0)
    ip = pp.addVars(nI,nD,nS,lb=0.0)
    pp.addConstrs((gp.quicksum(a_i[i]*qa[i,d,s] for i in I)<=Cap_3DP*n3DP_sol[d] for d in D for s in S))
    pp.addConstrs((B_sol[i,d]+qp[i,d,s]+qa[i,d,s]-gp.quicksum(fp[i,d,c,s] for c in C)==ip[i,d,s]
                   for i in I for d in D for s in S))
    pp.addConstrs((gp.quicksum(fp[i,d,c,s] for d in D)+up[i,c,s]==d_ics[i,c,s]
                   for i in I for c in C for s in S))
    pp.addConstrs((fp[i,d,c,s]<=F_bar[i,d]*float(dc_sol[d,i]) for i in I for d in D for c in C for s in S))
    pp.addConstrs((qp[i,d,s]<=Q_bar_CM[i,d]*float(w_sol[i,d]>0.5) for i in I for d in D for s in S))
    pp.addConstrs((qa[i,d,s]<=Q_bar_AM[i,d]*float(v_sol[i,d]>0.5) for i in I for d in D for s in S))
    pp.setObjective(gp.quicksum(pi_s*(
        gp.quicksum(c_CM[i]*qp[i,d,s] for i in I for d in D)+
        gp.quicksum(prod_i[i]*qa[i,d,s] for i in I for d in D)+
        gp.quicksum(t_dic[d,i,c]*fp[i,d,c,s] for i in I for d in D for c in C)+
        gp.quicksum(h_d[d]*ip[i,d,s] for i in I for d in D)+
        gp.quicksum(b_i[i]*up[i,c,s] for i in I for c in C)) for s in S), GRB.MINIMIZE)
    pp.optimize()

    if pp.Status==GRB.OPTIMAL:
        cost_breakdown = {
            'base_stock': cost_BS_sol, 'EOQ_CM': cost_EOQ_sol,
            'EOQ_raw': sum(C_EOQ_raw_sol), 'holding_raw': sum(h_d[d]*c_raw*B_raw_sol_arr[d] for d in D),
            'leasing_3DP': cost_leas_sol,
            'backorder_LT_analitico': bo_lt_sol,
            'trasporto': sum(pi_s*t_dic[d,i,c]*fp[i,d,c,s].X for i in I for d in D for c in C for s in S),
            'produzione_CM': sum(pi_s*c_CM[i]*qp[i,d,s].X for i in I for d in D for s in S),
            'produzione_AM': sum(pi_s*prod_i[i]*qa[i,d,s].X for i in I for d in D for s in S),
            'holding_inv': sum(pi_s*h_d[d]*ip[i,d,s].X for i in I for d in D for s in S),
            'backorder_2ndstage': sum(pi_s*b_i[i]*up[i,c,s].X for i in I for c in C for s in S),
        }
        obj_real = cost_1st+pp.ObjVal
        if abs(obj_real - mdl.ObjVal) > 1.0:
            print(f"  [ATTENZIONE] obj_real ({obj_real:,.2f}) != mdl.ObjVal "
                  f"({mdl.ObjVal:,.2f}), diff={obj_real-mdl.ObjVal:,.2f}€")
    else:
        cost_breakdown = {}
        obj_real = mdl.ObjVal

    return {'status':status,'obj':mdl.ObjVal,'obj_real':obj_real,
            'gap':mdl.MIPGap*100,'time':solve_time,
            'am':am_sol,'cm':cm_sol,'dc':dc_sol,'B':B_sol,
            'n3DP':n3DP_sol,'n_dc_per_sku':n_dc_per_sku,
            'cost_breakdown':cost_breakdown,'y_sol':y_sol}


def print_results(res, name, nI, nD, c_CM, c_AM, b_i, L_CM, L_AM):
    print(f"\n{'='*70}\nRISULTATI: {name}\n{'='*70}")
    if res['status']=='INFEASIBLE': print("  INFEASIBLE"); return
    print(f"  Status: {res['status']}  |  Costo: {res['obj_real']:,.2f}€  |  "
          f"Gap: {res['gap']:.2f}%  |  Tempo: {res['time']:.1f}s")
    print(f"\n  {'SKU':<4} {'T':<3} {'DC':<10} {'B':>4} {'c_CM':>7} {'c_AM':>7} {'ratio':>6}")
    print(f"  {'-'*55}")
    for i in range(nI):
        t = "AM" if res['am'][i] else "CM"
        dcs = [d for d in range(nD) if res['dc'][d,i]]
        bm = res['B'][i,dcs].mean() if dcs else 0
        print(f"  {i:<4} {t:<3} {str(dcs):<10} {bm:>4.1f} {c_CM[i]:>7.0f} "
              f"{c_AM[i]:>7.0f} {c_AM[i]/c_CM[i]:>6.2f}")
    print(f"\n  Stampanti: {np.round(res['n3DP'],1)}")
    if res.get('cost_breakdown'):
        print(f"  Costi:")
        tot = 0.0
        for k,v in res['cost_breakdown'].items():
            if abs(v)>0.01:
                print(f"    {k:<24}: {v:>12.2f}€")
                tot += v
        print(f"    {'TOTALE':<24}: {tot:>12.2f}€")


# ══════════════════════════════════════════════════════════════════════════
# CASO 6 SKU — "AM vince per lead time, non per prezzo"
# ══════════════════════════════════════════════════════════════════════════
nI, nD, nC = 6, 3, 5
seed = 42

coord_DC = np.array([[1.0,8.0],[1.0,2.0],[4.0,5.0]])
coord_C  = np.array([[1.5,9.0],[0.5,7.5],[1.5,1.0],[0.5,2.5],[4.5,5.5]])

c_CM = np.array([5000.,5000., 1000.,1000., 200.,200.])
c_AM = np.array([6000.,6000., 1500.,1500., 800.,800.])
n_i  = c_AM / c_CM

L_CM = np.array([26,26, 26,26, 9,9]) / 52
L_AM_vec = np.array([1,1, 1,1, 2,2]) / 52
L_AM = np.column_stack([L_AM_vec] * nD)

b_i  = np.array([150000.,120000., 50000.,40000., 3000.,2000.])
SL_i = np.array([0.99,0.99, 0.99,0.99, 0.90,0.88])

mu_ic = np.array([
    [0.8,0.7,0.5,0.5,0.5],
    [0.6,0.6,0.6,0.6,0.6],
    [0.5,0.4,0.4,0.4,0.3],
    [0.4,0.4,0.4,0.4,0.4],
    [2.0,2.0,2.0,2.0,2.0],
    [2.0,2.0,2.0,2.0,2.0],
])
alpha_ic = np.column_stack([SL_i] * nC)
a_i = c_AM / (1.30 * 0.00525 * 3600)
h_d = np.full(nD, 0.20)

costo_km = 50.0
Leas     = 15000.0

dist = np.array([[np.sqrt((coord_DC[d,0]-coord_C[c,0])**2 +
                          (coord_DC[d,1]-coord_C[c,1])**2)
                  for c in range(nC)] for d in range(nD)])
t_dic = np.stack([dist * costo_km] * nI, axis=1)

kwargs = dict(nI=nI, nD=nD, nC=nC, mu_ic=mu_ic, alpha_ic=alpha_ic,
    SL=SL_i, SL_raw=0.85, c_CM=c_CM, o=50., h_d=h_d, b_i=b_i, L_CM=L_CM,
    n_i=n_i, c_raw=50., den_raw=1200., unit_raw=0.5,
    o_raw=30., L_raw=1/52, L_AM=L_AM, Leas=Leas, Cap_3DP=5000., a_i=a_i,
    t_central=0., t_dic=t_dic, xi=2.0, seed=seed)

# ── RUN A: SENZA costo backorder-LT (modello originale) ──────────────────
print("="*72)
print("RUN A — SENZA costo backorder analitico da lead time (baseline)")
print("="*72)
res_A = solve_stochastic(**kwargs, nS=30, mip_gap=0.005, time_limit=300,
                         include_backorder_LT=False, verbose=True,
                         scenario_name="6SKU_no_BO_LT")
print_results(res_A, "SENZA backorder-LT", nI, nD, c_CM, c_AM, b_i, L_CM, L_AM)

# ── RUN B: CON costo backorder-LT (modello corretto) ──────────────────────
print("\n"+"="*72)
print("RUN B — CON costo backorder analitico da lead time (corretto)")
print("="*72)
res_B = solve_stochastic(**kwargs, nS=30, mip_gap=0.005, time_limit=300,
                         include_backorder_LT=True, verbose=True,
                         scenario_name="6SKU_with_BO_LT")
print_results(res_B, "CON backorder-LT", nI, nD, c_CM, c_AM, b_i, L_CM, L_AM)

# ── CONFRONTO ──────────────────────────────────────────────────────────────
print("\n"+"="*72)
print("CONFRONTO A (senza) vs B (con costo backorder-LT)")
print("="*72)
am_A = [i for i in range(nI) if res_A['am'][i]] if res_A['am'] is not None else []
am_B = [i for i in range(nI) if res_B['am'][i]] if res_B['am'] is not None else []
print(f"\n  {'':>20} {'AM':>15} {'Costo':>14} {'B medio':>10}")
print(f"  {'-'*62}")
print(f"  {'A - senza BO-LT':>20} {str(am_A):>15} {res_A['obj_real']:>14,.0f} "
      f"{res_A['B'].mean():>10.2f}")
print(f"  {'B - con BO-LT':>20} {str(am_B):>15} {res_B['obj_real']:>14,.0f} "
      f"{res_B['B'].mean():>10.2f}")
if am_A != am_B:
    print(f"\n  Il costo backorder-LT ha CAMBIATO la decisione tecnologica:")
    for i in range(nI):
        ta = "AM" if i in am_A else "CM"
        tb = "AM" if i in am_B else "CM"
        if ta != tb:
            print(f"    SKU {i}: A={ta} -> B={tb}")
else:
    print(f"\n  Stessa decisione tecnologica. Il costo backorder-LT ha "
          f"solo ridistribuito il costo totale (+{res_B['obj_real']-res_A['obj_real']:,.0f}€).")

RUN A — SENZA costo backorder analitico da lead time (baseline)
Set parameter Username
Set parameter LicenseID to value 2800035
Academic license - for non-commercial use only - expires 2027-03-30
Set parameter OutputFlag to value 1
Set parameter MIPGap to value 0.005
Set parameter TimeLimit to value 300
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-1165G7 @ 2.80GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  300
MIPGap  0.005

Optimize a model with 6054 rows, 5610 columns and 21243 nonzeros (Min)
Model fingerprint: 0xb8e6beb4
Model has 5283 linear objective coefficients
Model has 96 simple general constraints
  96 PWL
Variable types: 5448 continuous, 162 integer (159 binary)
Coefficient statistics:
  Matrix range     [4e-03, 9e+05]
  Objective range  [7e-03, 2e+04]
  Bounds range     [1e+00, 1e+00]
  R

#50

In [2]:
#!/usr/bin/env python3
"""
================================================================================
RAMP-ML — 6 SKU "AM vince per lead time" — CON COSTO BACKORDER ANALITICO
================================================================================
"""

import numpy as np
import gurobipy as gp
from gurobipy import GRB
from scipy.stats import poisson
from itertools import combinations
import time


def expected_backorder(mu_L, beta):
    """E[(D_L - beta)+] for D_L ~ Poisson(mu_L). Identica alla versione CG."""
    if mu_L < 1e-9 or beta < 0:
        return mu_L
    if beta == 0:
        return mu_L
    return max(mu_L*(1-poisson.cdf(beta-1, mu_L)) - beta*(1-poisson.cdf(beta, mu_L)), 0.0)


def solve_stochastic(
    nI, nD, nC, nS,
    mu_ic, alpha_ic,
    SL, SL_raw,
    c_CM, o, h_d, b_i, L_CM,
    n_i, c_raw, den_raw, unit_raw,
    o_raw, L_raw, L_AM, Leas, Cap_3DP, a_i,
    t_central,
    t_dic=None, B_cap=None, costo_km=100.0,
    coord_DC=None, coord_C=None,
    L_card=None, U_card=None,
    mip_gap=0.01, time_limit=600,
    xi=2.0, seed=42,
    warm_start_sol=None,
    verbose=False, scenario_name="default",
    include_backorder_LT=True,   # <-- TOGGLE per confronto A/B
):
    t_start = time.time()
    I, D, C, S = range(nI), range(nD), range(nC), range(nS)
    pi_s = 1.0 / nS
    SL_i = np.full(nI, SL) if np.isscalar(SL) else np.asarray(SL, dtype=float)

    if t_dic is None and coord_DC is not None and coord_C is not None:
        dist = np.array([
            [np.sqrt((coord_DC[d,0]-coord_C[c,0])**2 +
                     (coord_DC[d,1]-coord_C[c,1])**2)
             for c in range(nC)] for d in range(nD)])
        t_dic = np.stack([dist * costo_km] * nI, axis=1)

    # ── Preprocessing ─────────────────────────────────────────────────────
    prod_i  = n_i * c_CM
    vol_i   = (prod_i / 1.30) * 1e-6
    qbar_i  = (vol_i * den_raw) / unit_raw
    ptime_i = prod_i / (1.30 * 0.00525 * 3600)

    def beta_poisson(mu_val, L_val, SL_target):
        lam = mu_val * L_val
        if lam <= 1e-9: return 0
        for s in range(100000):
            if poisson.cdf(s, lam) >= SL_target: return int(s)
        return 99999

    mu_grid_set = {0.0}
    for i in I:
        for r in range(1, nC + 1):
            for combo in combinations([float(mu_ic[i,c]) for c in C], r):
                mu_grid_set.add(round(sum(combo), 6))
    mu_grid = np.array(sorted(mu_grid_set))
    M = len(mu_grid)

    B_CM_grid  = np.zeros((nI, nD, M))
    B_AM_grid  = np.zeros((nI, nD, M))
    C_EOQ_grid = np.zeros((nI, nD, M))
    for i in I:
        for d in D:
            for m, mu_m in enumerate(mu_grid):
                B_CM_grid[i,d,m] = beta_poisson(mu_m, L_CM[i], SL_i[i])
                B_AM_grid[i,d,m] = beta_poisson(mu_m, L_AM[i,d], SL_i[i])
                if mu_m > 0:
                    C_EOQ_grid[i,d,m] = np.sqrt(2*o*h_d[d]*c_CM[i]*mu_m)

    B_bar = B_CM_grid[:,:,-1]
    mu_id_tot = mu_ic.sum(axis=1)
    mu_max_i = np.array([mu_ic[i,:].sum() for i in I])

    contributi_raw = [float(qbar_i[i]*mu_id_tot[i]) for i in I]
    if nI <= 15:
        mu_raw_grid_set = {0.0}
        for r in range(1, nI+1):
            for combo in combinations(contributi_raw, r):
                mu_raw_grid_set.add(round(sum(combo), 4))
        mu_raw_grid = np.array(sorted(mu_raw_grid_set))
    else:
        mu_raw_grid = np.linspace(0, sum(contributi_raw), 200)
    M_RAW = len(mu_raw_grid)
    B_raw_grid = np.zeros((nD, M_RAW))
    C_EOQ_raw_grid = np.zeros((nD, M_RAW))
    B_bar_raw = np.zeros(nD)
    for d in D:
        for m, mu_m in enumerate(mu_raw_grid):
            B_raw_grid[d,m] = beta_poisson(mu_m, L_raw, SL_raw)
            if mu_m > 0:
                C_EOQ_raw_grid[d,m] = np.sqrt(2*o_raw*h_d[d]*c_raw*mu_m)
        B_bar_raw[d] = B_raw_grid[d,-1]

    rng = np.random.default_rng(seed)
    d_ics = rng.poisson(mu_ic[:,:,np.newaxis], size=(nI,nC,nS)).astype(float)

    Q_bar_CM = np.zeros((nI,nD)); F_bar = np.zeros((nI,nD))
    for i in I:
        for d in D:
            Q_bar_CM[i,d] = d_ics[i,:,:].sum(axis=0).max() * xi
            F_bar[i,d] = d_ics[i,:,:].max() * xi
    Q_bar_AM = np.full((nI,nD), Cap_3DP * xi)

    if L_card is None: L_card = np.ones(nI, dtype=int)
    if U_card is None: U_card = np.full(nI, nD, dtype=int)

    # ── PWL breakpoints (comuni a beta_aux e ai nuovi bo_aux) ─────────────
    N_PWL = 30
    pwl_mu_pts = {}
    for i in I:
        pwl_mu_pts[i] = np.linspace(0.0, max(mu_max_i[i], 0.01), N_PWL)

    # ── Modello ───────────────────────────────────────────────────────────
    mdl = gp.Model(f"stoc_{scenario_name}")
    mdl.setParam("OutputFlag", 1 if verbose else 0)
    mdl.setParam("MIPGap", mip_gap)
    mdl.setParam("TimeLimit", time_limit)

    am   = mdl.addVars(nI, vtype=GRB.BINARY, name="am")
    cm   = mdl.addVars(nI, vtype=GRB.BINARY, name="cm")
    y    = mdl.addVars(nD, nI, nC, vtype=GRB.BINARY, name="y")
    dc   = mdl.addVars(nD, nI, vtype=GRB.BINARY, name="dc")
    v    = mdl.addVars(nI, nD, vtype=GRB.BINARY, name="v")
    w    = mdl.addVars(nI, nD, vtype=GRB.BINARY, name="w")
    B    = mdl.addVars(nI, nD, lb=0.0, name="B")
    n3DP = mdl.addVars(nD, vtype=GRB.INTEGER, lb=0, name="n3DP")
    mu_var = mdl.addVars(nD, nI, lb=0.0, name="mu_var")
    C_EOQ_var = mdl.addVars(nD, nI, lb=0.0, name="C_EOQ_var")
    dcAM = mdl.addVars(nD, vtype=GRB.BINARY, name="dcAM")
    C_EOQ_raw_var = mdl.addVars(nD, lb=0.0, name="C_EOQ_raw_var")
    B_raw = mdl.addVars(nD, lb=0.0, name="B_raw")
    mu_raw_contrib = mdl.addVars(nD, nI, lb=0.0, name="mu_raw_contrib")
    mu_raw_total = mdl.addVars(nD, lb=0.0, name="mu_raw_total")
    beta_cm_aux = mdl.addVars(nI, nD, lb=0.0, name="beta_cm_aux")
    beta_am_aux = mdl.addVars(nI, nD, lb=0.0, name="beta_am_aux")

    # ══════════════════════════════════════════════════════════════════════
    # [NUOVO] Variabili ausiliarie PWL per il costo di backorder analitico
    # ══════════════════════════════════════════════════════════════════════
    bo_cm_aux = mdl.addVars(nI, nD, lb=0.0, name="bo_cm_aux")
    bo_am_aux = mdl.addVars(nI, nD, lb=0.0, name="bo_am_aux")

    q_CM  = mdl.addVars(nI, nD, nS, lb=0.0, name="q_CM")
    q_AM  = mdl.addVars(nI, nD, nS, lb=0.0, name="q_AM")
    f     = mdl.addVars(nI, nD, nC, nS, lb=0.0, name="f")
    u     = mdl.addVars(nI, nC, nS, lb=0.0, name="u")
    I_inv = mdl.addVars(nI, nD, nS, lb=0.0, name="I_inv")

    # ── Vincoli primo stadio ──────────────────────────────────────────────
    mdl.addConstrs((am[i]+cm[i]==1 for i in I), name="A1")
    mdl.addConstrs((y[d,i,c]<=dc[d,i] for d in D for i in I for c in C), name="B1")
    mdl.addConstrs((dc[d,i]<=gp.quicksum(y[d,i,c] for c in C) for d in D for i in I), name="B2")
    mdl.addConstrs((gp.quicksum(y[d,i,c] for d in D)==1 for i in I for c in C), name="assign")
    mdl.addConstrs((gp.quicksum(dc[d,i] for d in D)>=L_card[i] for i in I), name="B3_lo")
    mdl.addConstrs((gp.quicksum(dc[d,i] for d in D)<=U_card[i] for i in I), name="B3_up")
    mdl.addConstrs((v[i,d]<=am[i] for i in I for d in D), name="D1")
    mdl.addConstrs((v[i,d]<=dc[d,i] for i in I for d in D), name="D2")
    mdl.addConstrs((v[i,d]>=am[i]+dc[d,i]-1 for i in I for d in D), name="D2b")
    mdl.addConstrs((w[i,d]<=cm[i] for i in I for d in D), name="W1")
    mdl.addConstrs((w[i,d]<=dc[d,i] for i in I for d in D), name="W2")
    mdl.addConstrs((w[i,d]>=cm[i]+dc[d,i]-1 for i in I for d in D), name="W3")

    mdl.addConstrs((mu_var[d,i]==gp.quicksum(mu_ic[i,c]*y[d,i,c] for c in C)
                    for d in D for i in I), name="DG2_direct")
    mdl.addConstrs((mu_var[d,i]<=mu_max_i[i]*dc[d,i] for d in D for i in I), name="DG2_dc")

    for i in I:
        pts = pwl_mu_pts[i].tolist()
        for d in D:
            bcm = [float(beta_poisson(mu, L_CM[i], SL_i[i])) for mu in pts]
            bam = [float(beta_poisson(mu, L_AM[i,d], SL_i[i])) for mu in pts]
            eoq = [float(np.sqrt(2*o*h_d[d]*c_CM[i]*mu)) if mu>0 else 0.0 for mu in pts]
            mdl.addGenConstrPWL(mu_var[d,i], beta_cm_aux[i,d], pts, bcm, name=f"PWL_bCM_{d}_{i}")
            mdl.addGenConstrPWL(mu_var[d,i], beta_am_aux[i,d], pts, bam, name=f"PWL_bAM_{d}_{i}")
            mdl.addGenConstrPWL(mu_var[d,i], C_EOQ_var[d,i], pts, eoq, name=f"PWL_EOQ_{d}_{i}")

            bo_cm_pts = [b_i[i] * expected_backorder(pts[m]*L_CM[i], bcm[m])
                         for m in range(len(pts))]
            bo_am_pts = [b_i[i] * expected_backorder(pts[m]*L_AM[i,d], bam[m])
                         for m in range(len(pts))]
            mdl.addGenConstrPWL(mu_var[d,i], bo_cm_aux[i,d], pts, bo_cm_pts,
                                name=f"PWL_boCM_{d}_{i}")
            mdl.addGenConstrPWL(mu_var[d,i], bo_am_aux[i,d], pts, bo_am_pts,
                                name=f"PWL_boAM_{d}_{i}")

    mdl.addConstrs((B[i,d]>=beta_cm_aux[i,d]-B_bar[i,d]*(1-cm[i]) for i in I for d in D), name="DG4a")
    mdl.addConstrs((B[i,d]>=beta_am_aux[i,d]-B_bar[i,d]*(1-am[i]) for i in I for d in D), name="DG4b")
    mdl.addConstrs((B[i,d]<=B_bar[i,d]*dc[d,i] for i in I for d in D), name="DG5")

    if B_cap is not None:
        mdl.addConstrs((gp.quicksum(B[i,d] for i in I)<=B_cap for d in D), name="DG5b")

    for d in D:
        for i in I: mdl.addConstr(dcAM[d]>=v[i,d], name=f"dcAM_lo_{d}_{i}")
        mdl.addConstr(dcAM[d]<=gp.quicksum(v[i,d] for i in I), name=f"dcAM_up_{d}")

    M_raw_big = float(max(qbar_i)*sum(mu_ic[i,:].sum() for i in I))*2
    for d in D:
        for i in I:
            mdl.addConstr(mu_raw_contrib[d,i]<=M_raw_big*v[i,d], name=f"RC1_{d}_{i}")
            mdl.addConstr(mu_raw_contrib[d,i]<=qbar_i[i]*mu_var[d,i], name=f"RC2_{d}_{i}")
            mdl.addConstr(mu_raw_contrib[d,i]>=qbar_i[i]*mu_var[d,i]-M_raw_big*(1-v[i,d]), name=f"RC3_{d}_{i}")
        mdl.addConstr(mu_raw_total[d]==gp.quicksum(mu_raw_contrib[d,i] for i in I), name=f"mu_raw_tot_{d}")
        mdl.addConstr(mu_raw_total[d]<=M_raw_big*dcAM[d], name=f"mu_raw_dc_{d}")

    mu_raw_max = float(sum(qbar_i[i]*mu_ic[i,:].sum() for i in I))
    mu_raw_pts = np.linspace(0.0, max(mu_raw_max,0.01), 30).tolist()
    for d in D:
        brp = [float(beta_poisson(mu, L_raw, SL_raw)) for mu in mu_raw_pts]
        erp = [float(np.sqrt(2*o_raw*h_d[d]*c_raw*mu)) if mu>0 else 0.0 for mu in mu_raw_pts]
        bra = mdl.addVar(lb=0.0, name=f"b_raw_aux_{d}")
        mdl.addGenConstrPWL(mu_raw_total[d], bra, mu_raw_pts, brp, name=f"PWL_Braw_{d}")
        mdl.addConstr(B_raw[d]>=bra, name=f"ER4_{d}")
        mdl.addConstr(B_raw[d]<=B_bar_raw[d]*dcAM[d], name=f"ER5_{d}")
        mdl.addGenConstrPWL(mu_raw_total[d], C_EOQ_raw_var[d], mu_raw_pts, erp, name=f"PWL_EOQraw_{d}")

    M_EOQ_big = float(C_EOQ_grid.max())*2
    C_EOQ_act = mdl.addVars(nD, nI, lb=0.0, name="C_EOQ_act")
    for d in D:
        for i in I:
            mdl.addConstr(C_EOQ_act[d,i]<=M_EOQ_big*w[i,d], name=f"L_EOQ1_{d}_{i}")
            mdl.addConstr(C_EOQ_act[d,i]<=C_EOQ_var[d,i], name=f"L_EOQ2_{d}_{i}")
            mdl.addConstr(C_EOQ_act[d,i]>=C_EOQ_var[d,i]-M_EOQ_big*(1-w[i,d]), name=f"L_EOQ3_{d}_{i}")

    M_BO_big = max(
        max(b_i[i] * mu_max_i[i] for i in I),
        1.0
    ) * 2
    BO_act = mdl.addVars(nD, nI, lb=0.0, name="BO_act")
    for d in D:
        for i in I:
            mdl.addConstr(BO_act[d,i] <= M_BO_big*cm[i] + M_BO_big*am[i], name=f"BO_cap_{d}_{i}")
            mdl.addConstr(BO_act[d,i] >= bo_cm_aux[i,d] - M_BO_big*(1-cm[i]), name=f"BO_lo_CM_{d}_{i}")
            mdl.addConstr(BO_act[d,i] >= bo_am_aux[i,d] - M_BO_big*(1-am[i]), name=f"BO_lo_AM_{d}_{i}")
            mdl.addConstr(BO_act[d,i] <= bo_cm_aux[i,d] + M_BO_big*(1-cm[i]), name=f"BO_up_CM_{d}_{i}")
            mdl.addConstr(BO_act[d,i] <= bo_am_aux[i,d] + M_BO_big*(1-am[i]), name=f"BO_up_AM_{d}_{i}")
            mdl.addConstr(BO_act[d,i] <= M_BO_big*dc[d,i], name=f"BO_dc_{d}_{i}")

    ratio_pt = np.where(qbar_i>1e-9, ptime_i/qbar_i, 0.0)
    mdl.addConstrs((n3DP[d]>=gp.quicksum(ratio_pt[i]*mu_raw_contrib[d,i] for i in I)/Cap_3DP for d in D), name="E1")

    # ══════════════════════════════════════════════════════════════════════
    # McCormick per tecnologia-specifico Base Stock
    # ══════════════════════════════════════════════════════════════════════
    B_cm_lin = mdl.addVars(nI, nD, lb=0.0, name="B_cm_lin")
    B_am_lin = mdl.addVars(nI, nD, lb=0.0, name="B_am_lin")
    for i in I:
        for d in D:
            Bub = float(B_bar[i,d])
            mdl.addConstr(B_cm_lin[i,d] <= Bub * w[i,d])
            mdl.addConstr(B_cm_lin[i,d] <= B[i,d])
            mdl.addConstr(B_cm_lin[i,d] >= B[i,d] - Bub*(1-w[i,d]))
            mdl.addConstr(B_am_lin[i,d] <= Bub * v[i,d])
            mdl.addConstr(B_am_lin[i,d] <= B[i,d])
            mdl.addConstr(B_am_lin[i,d] >= B[i,d] - Bub*(1-v[i,d]))

    cost_BS = gp.quicksum(c_CM[i]*(1+h_d[d])*B_cm_lin[i,d] + prod_i[i]*(1+h_d[d])*B_am_lin[i,d] for i in I for d in D)
    cost_EOQ = gp.quicksum(C_EOQ_act[d,i] for i in I for d in D)
    # cost_raw RIMOSSO: consumo diretto materia prima gia' incluso in prod_i,
    # contarlo separatamente era doppio conteggio. Restano EOQ_raw/holding_raw.
    cost_raw_dc = gp.quicksum(C_EOQ_raw_var[d]+h_d[d]*c_raw*B_raw[d] for d in D)
    cost_leas = gp.quicksum(Leas*n3DP[d] for d in D)

    if include_backorder_LT:
        cost_backorder_LT = gp.quicksum(BO_act[d,i] for i in I for d in D)
    else:
        cost_backorder_LT = 0.0

    # Secondo stadio (invariato)
    mdl.addConstrs((gp.quicksum(a_i[i]*q_AM[i,d,s] for i in I)<=Cap_3DP*n3DP[d] for d in D for s in S), name="E2")
    mdl.addConstrs((B[i,d]+q_CM[i,d,s]+q_AM[i,d,s]-gp.quicksum(f[i,d,c,s] for c in C)==I_inv[i,d,s]
                    for i in I for d in D for s in S), name="F1")
    mdl.addConstrs((gp.quicksum(f[i,d,c,s] for d in D)+u[i,c,s]==d_ics[i,c,s]
                    for i in I for c in C for s in S), name="F2")
    mdl.addConstrs((f[i,d,c,s]<=F_bar[i,d]*dc[d,i] for i in I for d in D for c in C for s in S), name="G1")
    mdl.addConstrs((q_CM[i,d,s]<=Q_bar_CM[i,d]*w[i,d] for i in I for d in D for s in S), name="G2")
    mdl.addConstrs((q_AM[i,d,s]<=Q_bar_AM[i,d]*v[i,d] for i in I for d in D for s in S), name="G3")
    mdl.addConstrs((gp.quicksum(pi_s*gp.quicksum(f[i,d,c,s] for d in D) for s in S)
                    >=alpha_ic[i,c]*gp.quicksum(pi_s*d_ics[i,c,s] for s in S)
                    for i in I for c in C), name="H")

    cost_2nd = gp.quicksum(pi_s*(
        gp.quicksum(c_CM[i]*q_CM[i,d,s] for i in I for d in D)+
        gp.quicksum(prod_i[i]*q_AM[i,d,s] for i in I for d in D)+
        gp.quicksum(t_dic[d,i,c]*f[i,d,c,s] for i in I for d in D for c in C)+
        gp.quicksum(h_d[d]*I_inv[i,d,s] for i in I for d in D)+
        gp.quicksum(b_i[i]*u[i,c,s] for i in I for c in C)
    ) for s in S)

    mdl.setObjective(cost_BS+cost_EOQ+cost_raw_dc+cost_leas
                     +cost_backorder_LT+cost_2nd, GRB.MINIMIZE)

    # VarHint
    from scipy.stats import poisson as _pois
    def _bp(mu, L, sl):
        lam = mu*L
        if lam<1e-9: return 0
        for ss in range(10000):
            if _pois.cdf(ss, lam)>=sl: return ss
        return 9999
    for i in I:
        mu_c = float(mu_ic[i,:].sum())
        b_cm_h, b_am_h = _bp(mu_c, L_CM[i], SL_i[i]), _bp(mu_c, L_AM[i,0], SL_i[i])
        risp = c_CM[i]*(1+h_d[0])*(b_cm_h-b_am_h)
        bo_cm = b_i[i]*expected_backorder(mu_c*L_CM[i], b_cm_h)
        bo_am = b_i[i]*expected_backorder(mu_c*L_AM[i,0], b_am_h)
        risp += (bo_cm - bo_am) if include_backorder_LT else 0.0
        costo = (prod_i[i]-c_CM[i])*mu_c
        netto = risp - costo
        if netto>0:
            am[i].VarHintVal=1.0; am[i].VarHintPri=10; cm[i].VarHintVal=0.0
        elif -Leas<netto<=0:
            am[i].VarHintVal=1.0; am[i].VarHintPri=5; cm[i].VarHintVal=0.0
        else:
            am[i].VarHintVal=0.0; am[i].VarHintPri=1; cm[i].VarHintVal=1.0
    mdl.update()

    mdl.optimize()
    solve_time = time.time() - t_start

    if mdl.Status not in [GRB.OPTIMAL, GRB.TIME_LIMIT, GRB.SUBOPTIMAL]:
        return {'status':'INFEASIBLE','obj':None,'obj_real':None,'gap':None,
                'time':solve_time,'am':None,'cm':None,'dc':None,'B':None,
                'n3DP':None,'cost_breakdown':None,'y_sol':None}

    status = {GRB.OPTIMAL:'OPTIMAL',GRB.TIME_LIMIT:'TIME_LIMIT',
              GRB.SUBOPTIMAL:'SUBOPTIMAL'}.get(mdl.Status, str(mdl.Status))

    def snap(val): return val if val>0.5 else 0.0
    am_sol = np.array([am[i].X>0.5 for i in I])
    cm_sol = np.array([cm[i].X>0.5 for i in I])
    dc_sol = np.array([[dc[d,i].X>0.5 for i in I] for d in D])
    B_sol  = np.array([[B[i,d].X for d in D] for i in I])
    n3DP_sol = np.array([max(0.0, n3DP[d].X) for d in D])
    w_sol = np.array([[w[i,d].X for d in D] for i in I])
    v_sol = np.array([[v[i,d].X for d in D] for i in I])
    y_sol = np.array([[[snap(y[d,i,c].X) for c in C] for i in I] for d in D])
    n_dc_per_sku = dc_sol.sum(axis=0)

    bo_lt_sol = sum(BO_act[d,i].X for i in I for d in D)

    mu_var_sol = np.array([[sum(mu_ic[i,c]*y_sol[d,i,c] for c in C) for i in I] for d in D])
    C_EOQ_act_sol = np.zeros((nD,nI))
    for d in D:
        for i in I:
            if w_sol[i,d]>0.5:
                m = int(np.argmin(np.abs(mu_grid-mu_var_sol[d,i])))
                C_EOQ_act_sol[d,i] = C_EOQ_grid[i,d,m]
    mu_raw_sol = np.array([[qbar_i[i]*mu_var_sol[d,i]*float(v_sol[i,d]>0.5) for i in I] for d in D])
    dcAM_sol = np.array([any(v_sol[i,d]>0.5 for i in I) for d in D])
    C_EOQ_raw_sol = np.zeros(nD); B_raw_sol_arr = np.zeros(nD)
    for d in D:
        if dcAM_sol[d]:
            mu_raw_d = mu_raw_sol[d,:].sum()
            m = int(np.argmin(np.abs(mu_raw_grid-mu_raw_d)))
            C_EOQ_raw_sol[d] = C_EOQ_raw_grid[d,m]; B_raw_sol_arr[d] = B_raw_grid[d,m]

    # FIX: base-stock tecnologia-specifico anche nel reporting (prima usava
    # sempre c_CM anche per gli SKU AM, disallineato dal vero obiettivo
    # che usa B_cm_lin/B_am_lin). w_sol/v_sol selezionano il ramo corretto.
    cost_BS_sol = sum(
        c_CM[i]*(1+h_d[d])*B_sol[i,d] if w_sol[i,d] > 0.5
        else (prod_i[i]*(1+h_d[d])*B_sol[i,d] if v_sol[i,d] > 0.5 else 0.0)
        for i in I for d in D
    )
    cost_EOQ_sol = sum(C_EOQ_act_sol[d,i] for d in D for i in I)
    # cost_raw_sol RIMOSSO (stesso motivo del fix nell'obiettivo)
    cost_raw_dc_sol = sum(C_EOQ_raw_sol[d]+h_d[d]*c_raw*B_raw_sol_arr[d] for d in D)
    cost_leas_sol = sum(Leas*n3DP_sol[d] for d in D)
    cost_1st = cost_BS_sol+cost_EOQ_sol+cost_raw_dc_sol+cost_leas_sol+bo_lt_sol

    pp = gp.Model("pp"); pp.setParam("OutputFlag",0)
    qp = pp.addVars(nI,nD,nS,lb=0.0); qa = pp.addVars(nI,nD,nS,lb=0.0)
    fp = pp.addVars(nI,nD,nC,nS,lb=0.0); up = pp.addVars(nI,nC,nS,lb=0.0)
    ip = pp.addVars(nI,nD,nS,lb=0.0)
    pp.addConstrs((gp.quicksum(a_i[i]*qa[i,d,s] for i in I)<=Cap_3DP*n3DP_sol[d] for d in D for s in S))
    pp.addConstrs((B_sol[i,d]+qp[i,d,s]+qa[i,d,s]-gp.quicksum(fp[i,d,c,s] for c in C)==ip[i,d,s]
                   for i in I for d in D for s in S))
    pp.addConstrs((gp.quicksum(fp[i,d,c,s] for d in D)+up[i,c,s]==d_ics[i,c,s]
                   for i in I for c in C for s in S))
    pp.addConstrs((fp[i,d,c,s]<=F_bar[i,d]*float(dc_sol[d,i]) for i in I for d in D for c in C for s in S))
    pp.addConstrs((qp[i,d,s]<=Q_bar_CM[i,d]*float(w_sol[i,d]>0.5) for i in I for d in D for s in S))
    pp.addConstrs((qa[i,d,s]<=Q_bar_AM[i,d]*float(v_sol[i,d]>0.5) for i in I for d in D for s in S))
    pp.setObjective(gp.quicksum(pi_s*(
        gp.quicksum(c_CM[i]*qp[i,d,s] for i in I for d in D)+
        gp.quicksum(prod_i[i]*qa[i,d,s] for i in I for d in D)+
        gp.quicksum(t_dic[d,i,c]*fp[i,d,c,s] for i in I for d in D for c in C)+
        gp.quicksum(h_d[d]*ip[i,d,s] for i in I for d in D)+
        gp.quicksum(b_i[i]*up[i,c,s] for i in I for c in C)) for s in S), GRB.MINIMIZE)
    pp.optimize()

    if pp.Status==GRB.OPTIMAL:
        cost_breakdown = {
            'base_stock': cost_BS_sol, 'EOQ_CM': cost_EOQ_sol,
            'EOQ_raw': sum(C_EOQ_raw_sol), 'holding_raw': sum(h_d[d]*c_raw*B_raw_sol_arr[d] for d in D),
            'leasing_3DP': cost_leas_sol,
            'backorder_LT_analitico': bo_lt_sol,
            'trasporto': sum(pi_s*t_dic[d,i,c]*fp[i,d,c,s].X for i in I for d in D for c in C for s in S),
            'produzione_CM': sum(pi_s*c_CM[i]*qp[i,d,s].X for i in I for d in D for s in S),
            'produzione_AM': sum(pi_s*prod_i[i]*qa[i,d,s].X for i in I for d in D for s in S),
            'holding_inv': sum(pi_s*h_d[d]*ip[i,d,s].X for i in I for d in D for s in S),
            'backorder_2ndstage': sum(pi_s*b_i[i]*up[i,c,s].X for i in I for c in C for s in S),
        }
        obj_real = cost_1st+pp.ObjVal
        # Controllo di coerenza: il totale ricalcolato dalla scomposizione
        # deve combaciare con l'obiettivo vero del solver (entro tolleranza
        # numerica). Se questo stampa un warning, c'e' un altro disallineamento
        # nella scomposizione da investigare prima di fidarsi dei numeri.
        if abs(obj_real - mdl.ObjVal) > 1.0:
            print(f"  [ATTENZIONE] obj_real ({obj_real:,.2f}) != mdl.ObjVal "
                  f"({mdl.ObjVal:,.2f}), diff={obj_real-mdl.ObjVal:,.2f}€")
    else:
        cost_breakdown = {}
        obj_real = mdl.ObjVal

    return {'status':status,'obj':mdl.ObjVal,'obj_real':obj_real,
            'gap':mdl.MIPGap*100,'time':solve_time,
            'am':am_sol,'cm':cm_sol,'dc':dc_sol,'B':B_sol,
            'n3DP':n3DP_sol,'n_dc_per_sku':n_dc_per_sku,
            'cost_breakdown':cost_breakdown,'y_sol':y_sol}


def print_results(res, name, nI, nD, c_CM, c_AM, b_i, L_CM, L_AM):
    print(f"\n{'='*70}\nRISULTATI: {name}\n{'='*70}")
    if res['status']=='INFEASIBLE': print("  INFEASIBLE"); return
    print(f"  Status: {res['status']}  |  Costo: {res['obj_real']:,.2f}€  |  "
          f"Gap: {res['gap']:.2f}%  |  Tempo: {res['time']:.1f}s")
    print(f"\n  {'SKU':<4} {'T':<3} {'DC':<10} {'B':>4} {'c_CM':>7} {'c_AM':>7} {'ratio':>6}")
    print(f"  {'-'*55}")
    for i in range(nI):
        t = "AM" if res['am'][i] else "CM"
        dcs = [d for d in range(nD) if res['dc'][d,i]]
        bm = res['B'][i,dcs].mean() if dcs else 0
        print(f"  {i:<4} {t:<3} {str(dcs):<10} {bm:>4.1f} {c_CM[i]:>7.0f} "
              f"{c_AM[i]:>7.0f} {c_AM[i]/c_CM[i]:>6.2f}")
    print(f"\n  Stampanti: {np.round(res['n3DP'],1)}")
    if res.get('cost_breakdown'):
        print(f"  Costi:")
        tot = 0.0
        for k,v in res['cost_breakdown'].items():
            if abs(v)>0.01:
                print(f"    {k:<24}: {v:>12.2f}€")
                tot += v
        print(f"    {'TOTALE':<24}: {tot:>12.2f}€")


# ══════════════════════════════════════════════════════════════════════════
# CASO 6 SKU — "AM vince per lead time, non per prezzo"
# ══════════════════════════════════════════════════════════════════════════
nI, nD, nC = 6, 3, 5
seed = 42

coord_DC = np.array([[1.0,8.0],[1.0,2.0],[4.0,5.0]])
coord_C  = np.array([[1.5,9.0],[0.5,7.5],[1.5,1.0],[0.5,2.5],[4.5,5.5]])

c_CM = np.array([5000.,5000., 1000.,1000., 200.,200.])
c_AM = np.array([6000.,6000., 1500.,1500., 800.,800.])
n_i  = c_AM / c_CM

L_CM = np.array([26,26, 26,26, 9,9]) / 52
L_AM_vec = np.array([1,1, 1,1, 2,2]) / 52
L_AM = np.column_stack([L_AM_vec] * nD)

b_i  = np.array([150000.,120000., 50000.,40000., 3000.,2000.])
SL_i = np.array([0.99,0.99, 0.99,0.99, 0.90,0.88])

mu_ic = np.array([
    [0.8,0.7,0.5,0.5,0.5],
    [0.6,0.6,0.6,0.6,0.6],
    [0.5,0.4,0.4,0.4,0.3],
    [0.4,0.4,0.4,0.4,0.4],
    [2.0,2.0,2.0,2.0,2.0],
    [2.0,2.0,2.0,2.0,2.0],
])
alpha_ic = np.column_stack([SL_i] * nC)
a_i = c_AM / (1.30 * 0.00525 * 3600)
h_d = np.full(nD, 0.20)

costo_km = 50.0
Leas     = 15000.0

dist = np.array([[np.sqrt((coord_DC[d,0]-coord_C[c,0])**2 +
                          (coord_DC[d,1]-coord_C[c,1])**2)
                  for c in range(nC)] for d in range(nD)])
t_dic = np.stack([dist * costo_km] * nI, axis=1)

kwargs = dict(nI=nI, nD=nD, nC=nC, mu_ic=mu_ic, alpha_ic=alpha_ic,
    SL=SL_i, SL_raw=0.85, c_CM=c_CM, o=50., h_d=h_d, b_i=b_i, L_CM=L_CM,
    n_i=n_i, c_raw=50., den_raw=1200., unit_raw=0.5,
    o_raw=30., L_raw=1/52, L_AM=L_AM, Leas=Leas, Cap_3DP=5000., a_i=a_i,
    t_central=0., t_dic=t_dic, xi=2.0, seed=seed)

# ── RUN A: SENZA costo backorder-LT (modello originale) ──────────────────
print("="*72)
print("RUN A — SENZA costo backorder analitico da lead time (baseline)")
print("="*72)
res_A = solve_stochastic(**kwargs, nS=50, mip_gap=0.005, time_limit=300,
                         include_backorder_LT=False, verbose=True,
                         scenario_name="6SKU_no_BO_LT")
print_results(res_A, "SENZA backorder-LT", nI, nD, c_CM, c_AM, b_i, L_CM, L_AM)

# ── RUN B: CON costo backorder-LT (modello corretto) ──────────────────────
print("\n"+"="*72)
print("RUN B — CON costo backorder analitico da lead time (corretto)")
print("="*72)
res_B = solve_stochastic(**kwargs, nS=50, mip_gap=0.005, time_limit=300,
                         include_backorder_LT=True, verbose=True,
                         scenario_name="6SKU_with_BO_LT")
print_results(res_B, "CON backorder-LT", nI, nD, c_CM, c_AM, b_i, L_CM, L_AM)

# ── CONFRONTO ──────────────────────────────────────────────────────────────
print("\n"+"="*72)
print("CONFRONTO A (senza) vs B (con costo backorder-LT)")
print("="*72)
am_A = [i for i in range(nI) if res_A['am'][i]] if res_A['am'] is not None else []
am_B = [i for i in range(nI) if res_B['am'][i]] if res_B['am'] is not None else []
print(f"\n  {'':>20} {'AM':>15} {'Costo':>14} {'B medio':>10}")
print(f"  {'-'*62}")
print(f"  {'A - senza BO-LT':>20} {str(am_A):>15} {res_A['obj_real']:>14,.0f} "
      f"{res_A['B'].mean():>10.2f}")
print(f"  {'B - con BO-LT':>20} {str(am_B):>15} {res_B['obj_real']:>14,.0f} "
      f"{res_B['B'].mean():>10.2f}")
if am_A != am_B:
    print(f"\n  Il costo backorder-LT ha CAMBIATO la decisione tecnologica:")
    for i in range(nI):
        ta = "AM" if i in am_A else "CM"
        tb = "AM" if i in am_B else "CM"
        if ta != tb:
            print(f"    SKU {i}: A={ta} -> B={tb}")
else:
    print(f"\n  Stessa decisione tecnologica. Il costo backorder-LT ha "
          f"solo ridistribuito il costo totale (+{res_B['obj_real']-res_A['obj_real']:,.0f}€).")

RUN A — SENZA costo backorder analitico da lead time (baseline)
Set parameter OutputFlag to value 1
Set parameter MIPGap to value 0.005
Set parameter TimeLimit to value 300
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-1165G7 @ 2.80GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  300
MIPGap  0.005

Optimize a model with 9594 rows, 9090 columns and 34143 nonzeros (Min)
Model fingerprint: 0xf33276a1
Model has 8763 linear objective coefficients
Model has 96 simple general constraints
  96 PWL
Variable types: 8928 continuous, 162 integer (159 binary)
Coefficient statistics:
  Matrix range     [4e-03, 9e+05]
  Objective range  [4e-03, 2e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [3e-01, 9e+05]
  PWLCon x range   [7e-02, 1e+02]
  PWLCon y range   [0e+00, 2e+03]

Using variable hints.
Presolve r